# Python Coding Problems for Data Engineers

A hands-on workbook of the Python that actually gets tested in data-engineering interviews: streaming a file bigger than RAM, deduplicating at-least-once deliveries, hash joins, schema drift, sessionization, rolling metrics, and the small utility classes (LRU cache, retry with backoff) that show up in production pipelines.

It covers **14 problems**. Every problem is broken into small cells: an explanation, the solution, then separate tests for the happy path, the edge cases, and the exact stated output.

## How to use this notebook

- **Run cells top to bottom** (`Kernel -> Restart & Run All`). Every problem is: a statement, small solution cells, then several test cells and a `**Check yourself**` cell.
- **Every solution is followed by self-checking assertions.** A cell that finishes without an exception is proof the solution is correct on the documented cases; the printed output shows the real results.
- **Standard library only.** No pandas, no PySpark, no magics and no shell escapes. Everything is deterministic and re-runnable.
- **Try before you look.** Cover the solution cells, attempt the problem, then reveal them and run the tests.
- **Notes cells** give the complexity, the edge cases, and 1-2 interview follow-ups you should be able to answer out loud.

## The problems

1. Deduplicate, count, and group without pandas
2. Multi-key sorting with key functions
3. Top-K and top-N-per-group with `heapq`
4. Rolling windows: max sum, moving average, percentile
5. Two pointers and prefix sums
6. Merge overlapping intervals
7. Binary search over sorted partitions
8. Generator-based streaming of a large file
9. Parse CSV, JSON, and NDJSON
10. Flatten nested JSON into rows
11. In-memory hash join of two datasets
12. Schema drift and late-arriving dedupe
13. Sessionize clickstream events
14. Production utilities: LRU cache and retry with backoff

The notebook closes with `## Interview questions`: 12 likely questions with concise model answers, plus runnable code that verifies several of those answers.

### Kernel sanity check

Everything here uses the standard library only: no pandas, no numpy, no Spark. This first cell proves the kernel is alive and shows which Python you are on; interviewers expect you to know the version you target, because syntax such as `str | None` and `itertools.batched` depends on it.

In [1]:
import platform
import sys

print(f"Python {sys.version.split()[0]} on {platform.system()}")
print("Standard library only: no pandas / pyspark / shell magics required.")

Python 3.13.11 on Darwin
Standard library only: no pandas / pyspark / shell magics required.


## Problem 1 - Deduplicate, count, and group without pandas

**Statement.** Implement the workhorses of in-memory aggregation with the standard library only:

1. `dedupe_preserving_order(records, key)` - keep the **first** occurrence of each key, in input order.
2. `frequency_map(items)` - the equivalent of `COUNT(*) GROUP BY item`.
3. `group_by(records, key)` - bucket rows per key, preserving first-seen group order.
4. `aggregate(rows)` - SQL-style `COUNT`, `COUNT(DISTINCT)`, `SUM`, and `AVG` per group.

**Example.**

```text
records = [{"id":"a","v":1}, {"id":"b","v":2}, {"id":"a","v":3}, {"id":"c","v":4}]
dedupe   -> [{"id":"a","v":1}, {"id":"b","v":2}, {"id":"c","v":4}]
frequency_map(["us","eu","us"]) -> {"us": 2, "eu": 1}
aggregate([("us",10.0), ("us",30.0), ("us",None)])
         -> {"us": {"rows": 3, "distinct": 2, "total": 40.0, "avg": 13.333...}}
```

**Constraints.**

- O(n) time; no pandas; do not mutate the input records.
- Preserve input order for dedupe, and first-seen order for groups.
- Decide and document null behaviour: here a null *key* is dropped, while a null *value* counts as a row but is excluded from `SUM` and `COUNT(DISTINCT)` (SQL semantics).

### 1a. Dedupe preserving order

The whole problem is one pass with a `set` of keys already seen. A set gives average O(1) membership, so the pass is O(n) time and O(k) memory for k distinct keys -- the same shape as a Spark `dropDuplicates`, but with deterministic first-seen order rather than arbitrary order.

The pitfall is null handling: a `None` key is not a real business key (it would merge unrelated rows), so it is dropped rather than treated as one group. Note also that the records themselves are appended by reference and never copied or mutated.

In [2]:
from collections import Counter, defaultdict
from collections.abc import Iterable
from typing import Any


def dedupe_preserving_order(records: Iterable[dict[str, Any]], key: str = "id") -> list[dict[str, Any]]:
    """Keep the first record seen for each value of `key`, in input order."""
    seen: set[Any] = set()
    out: list[dict[str, Any]] = []
    for record in records:
        value = record.get(key)
        if value is None or value in seen:
            continue
        seen.add(value)
        out.append(record)
    return out

### 1b. `frequency_map` - `COUNT(*) GROUP BY item`

`collections.Counter` is a dict subclass specialised for counting, so the implementation is a one-liner that is still O(n) time and O(k) memory. Because `Counter` iterates in first-seen order, `most_common` and the raw dict both produce deterministic output -- important when a test or a dashboard compares two runs.

Keep the input out of the pipeline after this: `frequency_map` doubles as the reducer for a distributed word count once each partition has emitted its local counts.

In [3]:
def frequency_map(items: Iterable[Any]) -> dict[Any, int]:
    """COUNT(*) GROUP BY item, without pandas."""
    return dict(Counter(items))

### 1c. `group_by` - bucket rows per key

A `defaultdict(list)` plus one append per record is the in-memory `GROUP BY`. Because dicts preserve insertion order since Python 3.7, the resulting groups come out in first-seen order, not sorted -- a detail worth stating in an interview, since a downstream `ORDER BY` is then unnecessary.

The method retains the original row objects, so the memory cost is O(n), not O(k). Mutating a row inside a group mutates the caller's data.

In [4]:
def group_by(records: Iterable[dict[str, Any]], key: str) -> dict[Any, list[dict[str, Any]]]:
    """GROUP BY key -> list of rows, preserving first-seen group order."""
    groups: dict[Any, list[dict[str, Any]]] = defaultdict(list)
    for record in records:
        groups[record.get(key)].append(record)
    return dict(groups)

### 1d. `aggregate` - `COUNT`, `COUNT(DISTINCT)`, `SUM`, `AVG`

One accumulator dict per group holds the row count, a set for distinct values, and the running total. A `None` value increments `rows` (so `COUNT(*)` matches SQL) but is skipped for the distinct set and the total, which is exactly `COUNT(col)`/`SUM(col)` semantics.

The average is computed once at the end from the total, never by averaging averages. That is the single most common aggregation bug in pipelines, and it is why `AVG` is not additive across partitions.

In [5]:
def aggregate(rows: Iterable[tuple[Any, float | None]]) -> dict[Any, dict[str, float]]:
    """COUNT / COUNT(DISTINCT) / SUM / AVG grouped by the first tuple element."""
    buckets: dict[Any, dict[str, Any]] = defaultdict(lambda: {"rows": 0, "seen": set(), "total": 0.0})
    for group, value in rows:
        bucket = buckets[group]
        bucket["rows"] += 1
        if value is not None:
            bucket["seen"].add(value)
            bucket["total"] += value
    return {
        group: {
            "rows": bucket["rows"],
            "distinct": len(bucket["seen"]),
            "total": bucket["total"],
            "avg": bucket["total"] / bucket["rows"],
        }
        for group, bucket in buckets.items()
    }

### 1e. Happy path

This is the documented example, and it also pins the *order* of the result, not just its contents. For dedupe the first `"a"` row wins (`v = 1`), and the returned records are the very same objects from the input list.

In [6]:
records = [
    {"id": "a", "v": 1},
    {"id": "b", "v": 2},
    {"id": "a", "v": 3},
    {"id": "c", "v": 4},
    {"id": "b", "v": 5},
]
print("dedupe:", dedupe_preserving_order(records))
assert dedupe_preserving_order(records) == [
    {"id": "a", "v": 1},
    {"id": "b", "v": 2},
    {"id": "c", "v": 4},
]
assert dedupe_preserving_order(records, key="v") == records  # every v is unique

dedupe: [{'id': 'a', 'v': 1}, {'id': 'b', 'v': 2}, {'id': 'c', 'v': 4}]


### 1f. Happy path - groups and aggregates

`group_by` must return groups in first-seen order (`a`, `b`, `c`) and keep each group's rows in input order. The aggregate check uses the documented null behaviour: three rows for `us`, two distinct non-null values, total 40.0, average `40/3`, while `eu` has two rows but one distinct value and therefore average 5.0.

In [7]:
grouped = group_by(records, key="id")
print("group order:", list(grouped))
assert list(grouped) == ["a", "b", "c"]  # first-seen order, not sorted
assert [row["v"] for row in grouped["a"]] == [1, 3]

agg = aggregate([("us", 10.0), ("eu", 5.0), ("us", 30.0), ("eu", 5.0), ("us", None)])
print("aggregate:", agg)
assert agg["us"] == {"rows": 3, "distinct": 2, "total": 40.0, "avg": 40.0 / 3}
assert agg["eu"] == {"rows": 2, "distinct": 1, "total": 10.0, "avg": 5.0}

group order: ['a', 'b', 'c']
aggregate: {'us': {'rows': 3, 'distinct': 2, 'total': 40.0, 'avg': 13.333333333333334}, 'eu': {'rows': 2, 'distinct': 1, 'total': 10.0, 'avg': 5.0}}


### 1g. Edge cases - empty input, null keys, duplicates

Empty input must return empty containers, never raise. A `None` key is dropped by dedupe, and duplicate `None` keys collapse to nothing rather than to a group. `frequency_map` on an empty iterable returns `{}` because `Counter` over nothing is empty.

In [8]:
assert dedupe_preserving_order([]) == []
assert dedupe_preserving_order([{"id": None}, {"id": None}, {"id": 1}]) == [{"id": 1}]
assert frequency_map(["us", "eu", "us", "us", "eu"]) == {"us": 3, "eu": 2}
assert frequency_map([]) == {}
assert group_by([], key="id") == {}
assert aggregate([]) == {}

### 1h. Stated output check

The documented example prints exactly these lines. Run the cell and compare the printed values with the docstring above; the asserts are the machine-checkable version of the same claim.

In [9]:
print("dedupe   :", [row["id"] for row in dedupe_preserving_order(records)])
print("frequency:", frequency_map(["us", "eu", "us", "us", "eu"]))
print("aggregate:", aggregate([("us", 10.0), ("us", 30.0), ("us", None)]))
print("P1 checks passed")

dedupe   : ['a', 'b', 'c']
frequency: {'us': 3, 'eu': 2}
aggregate: {'us': {'rows': 3, 'distinct': 2, 'total': 40.0, 'avg': 13.333333333333334}}
P1 checks passed


**Complexity & edge cases.** Dedupe and counting are O(n) time; dedupe uses O(k) memory for k distinct keys, while `group_by`/`aggregate` use O(n) because they retain rows or distinct values. Edge cases: empty input returns `[]`/`{}`; unhashable keys must be normalised to tuples; `1 == True` and `1 == 1.0` collide in a Python dict, so normalise types on ingest; summing money in `float` drifts, so accumulate integer cents or `Decimal`; keep order in a list because iterating a `set` is not order-stable across runs.

**Interview follow-ups.**

1. *The data no longer fits in RAM - how does each operation change?* Sort-merge or hash-partition on disk; approximate distinct counts with HyperLogLog; a group-by becomes a shuffle in Spark.
2. *Why is `dict`/`set` lookup called O(1), and when is it not?* Average-case hashing; worst case O(n) with adversarial collisions; keys must be hashable with a stable hash.

## Problem 2 - Multi-key sorting with key functions

**Statement.** Reproduce `ORDER BY status ASC, amount DESC, id ASC`, then generalise to any number of `(field, descending)` pairs. Also handle nulls-last and the case where a key function cannot express the comparison.

**Example.**

```text
input:
  [{"status":"a","amount":3,"id":3}, {"status":"a","amount":1,"id":1},
   {"status":"b","amount":2,"id":2}, {"status":"a","amount":1,"id":4}]
output:
  [{"status":"a","amount":3,"id":3}, {"status":"a","amount":1,"id":1},
   {"status":"a","amount":1,"id":4}, {"status":"b","amount":2,"id":2}]
```

**Constraints.** Use a key function where possible; avoid `cmp_to_key` in the hot path; `None` must sort last and never raise `TypeError`; do not mutate the input list.

### 2a. One tuple key for a fixed `ORDER BY`

A tuple key is compared element by element, so `(status, -amount, id)` reproduces `status ASC, amount DESC, id ASC` in a single pass. Negating a numeric field is the cheapest way to flip its direction, and it works because the key is computed once per row (decorate-sort-undecorate) rather than once per comparison.

The catch is that `-x` only exists for numbers. Strings, dates, and custom objects cannot be negated, which is what forces the general version next.

In [10]:
from functools import cmp_to_key
from typing import Any


def sort_orders(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """ORDER BY status ASC, amount DESC, id ASC using a single tuple key."""
    return sorted(rows, key=lambda row: (row["status"], -row["amount"], row["id"]))

### 2b. Any number of mixed-direction keys

Python's Timsort is stable, so sorting repeatedly from the least-significant key to the most-significant one gives the same result as a single combined key. Each pass only has to handle one direction, so strings and dates work fine and no value is ever negated.

The cost is one extra pass per key and a copy of the list; for a handful of keys that is irrelevant next to the O(n log n) comparisons. Copy with `list(rows)` first so the caller's list is never reordered.

In [11]:
def sort_by_keys_stable(rows: list[dict[str, Any]], keys: list[tuple[str, bool]]) -> list[dict[str, Any]]:
    """ORDER BY any number of (field, descending) pairs using stable passes."""
    out = list(rows)
    for field, descending in reversed(keys):
        out.sort(key=lambda row: row[field], reverse=descending)
    return out

### 2c. Nulls last

Comparing `None` with an `int` raises `TypeError`, so a null flag must lead the tuple: `False` sorts before `True`, pushing every null to the end regardless of the value type. This is the standard `NULLS LAST` emulation and it works for any comparable column.

If the direction is descending, you place the flag accordingly (`row[field] is not None`) -- a common off-by-one in interview answers.

In [12]:
def sort_nulls_last(rows: list[dict[str, Any]], field: str) -> list[dict[str, Any]]:
    """None sorts after every real value because the boolean flag leads the key."""
    return sorted(rows, key=lambda row: (row[field] is None, row[field]))

### 2d. The comparator escape hatch

Sometimes the ordering is genuinely not expressible as a key -- for example custom business logic that compares two records field by field with early exit. `functools.cmp_to_key` wraps a `compare(a, b) -> -1/0/1` function into something `sorted` accepts.

It is the slow path: the comparator runs from Python on every comparison, so O(n log n) comparisons each pay interpreter overhead instead of one cheap key per element. Mention it as a fallback, not as the default.

In [13]:
def sort_with_comparator(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """The escape hatch when a key function cannot express the comparison."""

    def compare(left: dict[str, Any], right: dict[str, Any]) -> int:
        if left["status"] != right["status"]:
            return -1 if left["status"] < right["status"] else 1
        if left["amount"] != right["amount"]:
            return -1 if left["amount"] > right["amount"] else 1
        return (left["id"] > right["id"]) - (left["id"] < right["id"])

    return sorted(rows, key=cmp_to_key(compare))

### 2e. Happy path

All three strategies must agree on the documented example. That agreement is the real test: the tuple key, the stable multi-pass sort, and the comparator are three implementations of one `ORDER BY`, so any divergence is a bug in the translation.

In [14]:
rows = [
    {"status": "a", "amount": 3, "id": 3},
    {"status": "a", "amount": 1, "id": 1},
    {"status": "b", "amount": 2, "id": 2},
    {"status": "a", "amount": 1, "id": 4},
]
expected = [
    {"status": "a", "amount": 3, "id": 3},
    {"status": "a", "amount": 1, "id": 1},
    {"status": "a", "amount": 1, "id": 4},
    {"status": "b", "amount": 2, "id": 2},
]
print("sorted:", [(r["status"], r["amount"], r["id"]) for r in sort_orders(rows)])
assert sort_orders(rows) == expected
assert sort_by_keys_stable(rows, [("status", False), ("amount", True), ("id", False)]) == expected
assert sort_with_comparator(rows) == expected

sorted: [('a', 3, 3), ('a', 1, 1), ('a', 1, 4), ('b', 2, 2)]


### 2f. Edge cases - stability, nulls, no mutation

`sorted` copies and `sort_by_keys_stable` copies first, so the caller's list must be byte-for-byte unchanged after either call -- a real requirement in tests and in pipelines that reuse a batch. Stability means equal keys keep their input order, which is what makes deterministic tie-breaking possible. Nulls-last has to handle an empty input and a `None` sitting next to a number without raising.

In [15]:
before = [dict(row) for row in rows]
sort_orders(rows)
assert rows == before, "sorted() must not mutate the input"

# Stability: equal keys keep their original relative order.
letters = [("b", 1), ("a", 1), ("b", 0)]
assert sorted(letters, key=lambda pair: pair[1]) == [("b", 0), ("b", 1), ("a", 1)]

# Nulls-last never raises, even when None sits next to a number.
assert sort_nulls_last([{"x": 2}, {"x": None}, {"x": 1}], "x") == [{"x": 1}, {"x": 2}, {"x": None}]
assert sort_nulls_last([], "x") == []

### 2g. Stated output check

Print the sorted projection from the documented example and confirm the order by eye, then let the asserts confirm it mechanically. The cell also re-checks that the generic multi-key path produces the same sequence.

In [16]:
print("expected:", [(r["status"], r["amount"], r["id"]) for r in expected])
print("actual  :", [(r["status"], r["amount"], r["id"]) for r in sort_orders(rows)])
assert sort_orders(rows) == sort_by_keys_stable(rows, [("status", False), ("amount", True), ("id", False)])
print("P2 checks passed")

expected: [('a', 3, 3), ('a', 1, 1), ('a', 1, 4), ('b', 2, 2)]
actual  : [('a', 3, 3), ('a', 1, 1), ('a', 1, 4), ('b', 2, 2)]
P2 checks passed


**Complexity & edge cases.** Sorting is O(n log n) comparisons plus O(n) memory for `sorted` (`list.sort` is in place). Python's Timsort is stable, which is what makes multi-pass sorting and deterministic tie-breaking work. Edge cases: `None` inside a tuple key raises `TypeError` unless a null flag leads the tuple; negating fails for strings, dates, and custom objects, so prefer `reverse=` passes; naive and aware datetimes cannot be compared; `cmp_to_key` runs Python for every comparison and is several times slower than a key function.

**Interview follow-ups.**

1. *How would you sort 100 GB on one machine?* Sort each chunk, write it to disk, then k-way merge with `heapq.merge`; keep only one buffer per chunk in memory.
2. *Why does `sorted` need the key function to be cheap?* The key is computed once per element (decorate-sort-undecorate), but an expensive or non-deterministic key breaks that assumption; the comparator variant recomputes logic on every comparison.

## Problem 3 - Top-K and top-N-per-group with `heapq`

**Statement.** Return the K most frequent keys from a stream, and the largest N rows per group, without sorting everything. A full sort is O(n log n); a bounded heap is O(n log k) and stores only k elements.

**Example.**

```text
keys = ["a","b","a","c","a","b","d"], k = 2
top_k -> [("a", 3), ("b", 2)]

rows grouped by dept, top 2 salaries per dept, computed in one pass
eng: [("bob",150), ("ana",120)]
ops: [("ed",110),  ("di",100)]
```

**Constraints.** Return `[]` for `k <= 0`; ties must be deterministic; keep memory bounded per group; do not sort the full input.

### 3a. `top_k` with `Counter.most_common`

`Counter.most_common(k)` counts everything in O(n), then uses a heap to pick the k largest counts in O(m log k) for m distinct keys. It is the shortest correct answer, and for `k == 0` it returns an empty list rather than raising.

Counting still needs O(m) memory, so for a genuinely unbounded key space this is a sketch problem (Count-Min or Space-Saving), not a `Counter` problem.

In [17]:
import heapq
from collections import Counter, defaultdict
from collections.abc import Iterable
from typing import Any


def top_k(keys: Iterable[Any], k: int) -> list[tuple[Any, int]]:
    """K most common keys; Counter.most_common uses a heap internally."""
    return Counter(keys).most_common(k)

### 3b. Explicit bounded min-heap

Building the heap by hand shows the mechanism an interviewer wants: keep a *min*-heap of size k, push while it is not full, and only replace the root when the incoming count beats the smallest kept count. Memory is O(k) beyond the counter, and the final `sorted` is O(k log k).

The tie-break `(-count, str(key))` makes equal counts deterministic; without it, two runs could return different keys.

In [18]:
def top_k_bounded(keys: Iterable[Any], k: int) -> list[tuple[Any, int]]:
    """Explicit bounded min-heap: O(n log k) time, O(k) extra memory."""
    if k <= 0:
        return []
    heap: list[tuple[int, Any]] = []
    for key, count in Counter(keys).items():
        if len(heap) < k:
            heapq.heappush(heap, (count, key))
        elif count > heap[0][0]:
            heapq.heapreplace(heap, (count, key))
    return sorted(((key, count) for count, key in heap), key=lambda item: (-item[1], str(item[0])))

### 3c. Top-N per group in one pass

Keep one bounded heap per group key. Each row is pushed with its running index, so the heap orders by `(metric, index)`; popping from a min-heap drops the smallest metric and, on a tie, the earliest index. That yields a deterministic "largest N, ties broken by input order".

Memory is O(g x N) for g groups rather than O(n), which is the whole point: a per-department leaderboard should not materialise every row.

In [19]:
def top_n_per_group(
    rows: Iterable[dict[str, Any]], group: str, metric: str, n: int
) -> dict[Any, list[dict[str, Any]]]:
    """Largest n rows per group in one pass, with bounded memory per group."""
    if n <= 0:
        return {}
    heaps: dict[Any, list[tuple[float, int, dict[str, Any]]]] = defaultdict(list)
    for index, row in enumerate(rows):
        heap = heaps[row[group]]
        # The running index breaks metric ties deterministically.
        heapq.heappush(heap, (row[metric], index, row))
        if len(heap) > n:
            heapq.heappop(heap)
    return {
        group_key: [
            row for _value, _index, row in sorted(heap, key=lambda item: (-item[0], item[1]))
        ]
        for group_key, heap in heaps.items()
    }

### 3d. Happy path - top K

The counted example gives `a` three times, `b` twice, then `c` and `d` once each, so `k = 2` is `[("a", 3), ("b", 2)]`. The manual heap must return the same pairs, and asking for more keys than exist must return everything.

In [20]:
keys = ["a", "b", "a", "c", "a", "b", "d"]
print("top_k:", top_k(keys, 2))
assert top_k(keys, 2) == [("a", 3), ("b", 2)]
assert top_k_bounded(keys, 2) == [("a", 3), ("b", 2)]
assert sorted(top_k_bounded(keys, 10)) == sorted(top_k(keys, 10))

top_k: [('a', 3), ('b', 2)]


### 3e. Happy path - largest N per group

For `eng`, the two largest salaries are Bob at 150 and Ana at 120; for `ops`, Ed at 110 and Di at 100. The output rows are the original dicts, so the caller can read every column, and the order within a group is largest-first.

In [21]:
rows = [
    {"dept": "eng", "name": "ana", "salary": 120},
    {"dept": "eng", "name": "bob", "salary": 150},
    {"dept": "eng", "name": "cy", "salary": 90},
    {"dept": "ops", "name": "di", "salary": 100},
    {"dept": "ops", "name": "ed", "salary": 110},
]
top2 = top_n_per_group(rows, "dept", "salary", 2)
print("top2 eng:", [(r["name"], r["salary"]) for r in top2["eng"]])
assert [(r["name"], r["salary"]) for r in top2["eng"]] == [("bob", 150), ("ana", 120)]
assert [(r["name"], r["salary"]) for r in top2["ops"]] == [("ed", 110), ("di", 100)]

top2 eng: [('bob', 150), ('ana', 120)]


### 3f. Edge cases - `k <= 0`, empty input, ties

A non-positive `k` must return an empty result instead of raising or looping forever, and an empty input must not invent groups. Ties are the subtle case: two keys with the same count (or two rows with the same metric) need an explicit secondary key or the output is order-dependent, which breaks reproducibility.

In [22]:
assert top_k(keys, 0) == []
assert top_k_bounded(keys, 0) == []
assert top_n_per_group(rows, "dept", "salary", 0) == {}
assert top_n_per_group([], "dept", "salary", 2) == {}

# Ties fall back to input order (the running index), so the result is stable.
tied = [
    {"dept": "d", "name": "first", "salary": 10},
    {"dept": "d", "name": "second", "salary": 10},
    {"dept": "d", "name": "low", "salary": 1},
]
assert [r["name"] for r in top_n_per_group(tied, "dept", "salary", 2)["d"]] == ["first", "second"]

### 3g. Stated output check

Print the documented `top_k` result and the top two engineers, then assert the exact tuples from the statement. This is the line a reviewer will read first, so it should match the `Example` block character for character.

In [23]:
print("top_k ->", top_k(keys, 2))
print("eng   ->", [(r["name"], r["salary"]) for r in top2["eng"]])
print("ops   ->", [(r["name"], r["salary"]) for r in top2["ops"]])
print("P3 checks passed")

top_k -> [('a', 3), ('b', 2)]
eng   -> [('bob', 150), ('ana', 120)]
ops   -> [('ed', 110), ('di', 100)]
P3 checks passed


**Complexity & edge cases.** `heapq.nlargest(k, ...)`/`nsmallest` and `Counter.most_common(k)` are O(n log k) with O(k) memory. `top_n_per_group` is O(n log n) worst case but only O(g x n) memory for g groups. Edge cases: `k` larger than the number of distinct keys returns everything; ties need an explicit secondary key or the output is order-dependent; one heavy hitter dominates a `Counter`, so a Count-Min Sketch is the scalable answer; unhashable keys must be normalised first; `k <= 0` must not crash.

**Interview follow-ups.**

1. *When is `nlargest` worse than a full sort?* When k is close to n: O(n log k) approaches O(n log n) while adding heap overhead, so `sorted(...)[:k]` can win.
2. *How do you get the top K globally across partitions?* Each partition emits its local top K, then a single reducer merges K x partitions candidates with a bounded heap.

## Problem 4 - Rolling windows: max sum, moving average, percentile

**Statement.** Implement the windowed primitives behind dashboards, rate limiters, and SLOs:

1. `moving_average(values, window)` - trailing simple moving average, one value per full window.
2. `max_window_sum(values, window)` - largest sum of any contiguous window of exactly `window` elements.
3. `percentile(values, pct)` and `rolling_percentile(values, window, pct)` - linear-interpolation percentiles.
4. `trailing_event_count(timestamps, window)` - events in `(t - window, t]`, amortised O(1) per event.

**Example.**

```text
moving_average([1,2,3,4,5], 3)          -> [2.0, 3.0, 4.0]
max_window_sum([1,2,3,4,5], 3)          -> 12
percentile([10,12,13,15,20,100], 50)    -> 14.0
trailing_event_count([0,1,2,10,11], 3)  -> [1, 2, 3, 1, 2]
```

**Constraints.** O(1) update per element for the average and the count; document the window boundary; raise on a non-positive window; handle a window larger than the input.

### 4a. Moving average - sliding sum, not a re-sum

Add the incoming value and subtract the one leaving the window, so each step is O(1) instead of O(window). The output has exactly `len(values) - window + 1` entries: one value per *full* window, with the partial prefix discarded.

Because the running total is a float, long windows accumulate rounding error; `math.fsum` on an occasional reset, or Kahan summation, keeps it honest. A window larger than the input returns `[]` rather than a partial average.

In [24]:
import math
from collections import deque
from collections.abc import Sequence


def moving_average(values: Sequence[float], window: int) -> list[float]:
    """Trailing simple moving average, one output per full window."""
    if window <= 0:
        raise ValueError("window must be positive")
    if window > len(values):
        return []
    out: list[float] = []
    total = 0.0
    for index, value in enumerate(values):
        total += value
        if index >= window:
            total -= values[index - window]
        if index >= window - 1:
            out.append(total / window)
    return out

### 4b. Maximum window sum

The same sliding total, but keep the best value seen instead of emitting every window. This is the O(n) shape used for "busiest 5-minute bucket" style questions; the naive recompute is O(n x window).

Returning `None` (rather than `0` or `[]`) for an impossible window keeps the failure explicit. Negative values matter: initialising `best` to 0 would be wrong on an all-negative array, so seed it with the first window's sum.

In [25]:
def max_window_sum(values: Sequence[float], window: int) -> float | None:
    """Largest sum of any contiguous window of exactly `window` elements."""
    if window <= 0 or window > len(values):
        return None
    total = sum(values[:window])
    best = total
    for index in range(window, len(values)):
        total += values[index] - values[index - window]
        best = max(best, total)
    return best

### 4c. Percentile with linear interpolation

Sort once, map the percentile to a fractional position `(n - 1) * pct / 100`, then interpolate between the two neighbouring order statistics. This matches the numpy/`pandas` default, so the numbers agree with a notebook the interviewer may compare against.

Two pitfalls: `[]` has no percentile (raise), and `pct` outside `[0, 100]` is a caller bug, not something to clamp silently. On a handful of points, p99 is simply the maximum.

In [26]:
def percentile(values: Sequence[float], pct: float) -> float:
    """Linear-interpolation percentile (the numpy default); pct in [0, 100]."""
    if not values:
        raise ValueError("empty sample")
    if not 0 <= pct <= 100:
        raise ValueError("pct must be in [0, 100]")
    ordered = sorted(values)
    position = (len(ordered) - 1) * pct / 100
    lower = math.floor(position)
    upper = math.ceil(position)
    if lower == upper:
        return float(ordered[int(position)])
    weight = position - lower
    return ordered[lower] * (1 - weight) + ordered[upper] * weight

### 4d. Rolling percentile - the honest O(n x w log w) version

Re-sorting each window is the correct baseline to write first. It is deliberately not the fancy answer: an exact rolling quantile needs an order-statistic structure (two heaps or a balanced tree) or an approximate sketch, and saying so out loud is worth more than silently shipping a slow loop.

The slice `values[i : i + window]` builds a fresh list per window, so the memory churn is O(n x w) allocations over the run.

In [27]:
def rolling_percentile(values: Sequence[float], window: int, pct: float) -> list[float]:
    """Percentile of each trailing window (recomputes the sort per window)."""
    if window <= 0:
        raise ValueError("window must be positive")
    return [percentile(values[i : i + window], pct) for i in range(len(values) - window + 1)]

### 4e. Trailing event count - the rate-limiter window

A `deque` of live timestamps gives O(1) amortised work per event: append the new timestamp, then pop from the left while the oldest event is outside the window. Each timestamp is pushed and popped at most once across the whole run.

The boundary is documented as half-open on the left: an event at exactly `t - window` is dropped, so the window is `(t - window, t]`. Choosing that convention explicitly prevents the classic off-by-one in rate limiters.

In [28]:
def trailing_event_count(timestamps: Sequence[int], window: int) -> list[int]:
    """Deque-based count of events in (t - window, t]; amortised O(1) per event."""
    buffer: deque[int] = deque()
    out: list[int] = []
    for current in timestamps:
        buffer.append(current)
        while buffer and buffer[0] <= current - window:
            buffer.popleft()
        out.append(len(buffer))
    return out

### 4f. Happy path - all four primitives

Every documented example in one cell: the moving average, the best window sum, the interpolated median, and the trailing counts. The median of `[10, 12, 13, 15, 20, 100]` is `(13 + 15) / 2 = 14.0`, which is the interpolation case rather than a raw sample.

In [29]:
print("moving_average:", moving_average([1, 2, 3, 4, 5], 3))
assert moving_average([1, 2, 3, 4, 5], 3) == [2.0, 3.0, 4.0]
assert moving_average([1], 1) == [1.0]
print("max_window_sum:", max_window_sum([1, 2, 3, 4, 5], 3))
assert max_window_sum([1, 2, 3, 4, 5], 3) == 12
print("percentile p50:", percentile([10.0, 12.0, 13.0, 15.0, 20.0, 100.0], 50))
assert percentile([10.0, 12.0, 13.0, 15.0, 20.0, 100.0], 50) == 14.0
print("trailing_event_count:", trailing_event_count([0, 1, 2, 10, 11], 3))
assert trailing_event_count([0, 1, 2, 10, 11], 3) == [1, 2, 3, 1, 2]

moving_average: [2.0, 3.0, 4.0]
max_window_sum: 12
percentile p50: 14.0
trailing_event_count: [1, 2, 3, 1, 2]


### 4g. Happy path - interpolation details

`percentile([1, 2, 3, 4], 50)` is 2.5 because the fractional position lands halfway between the second and third samples; the endpoints (`pct = 0` and `pct = 100`) must return the min and max exactly. A single-value sample returns that value for any percentile.

In [30]:
assert percentile([1, 2, 3, 4], 50) == 2.5
assert percentile([5], 99) == 5.0
assert percentile([1.0, 2.0, 3.0], 0) == 1.0
assert percentile([1.0, 2.0, 3.0], 100) == 3.0
assert rolling_percentile([1, 2, 3, 4], 3, 100) == [3.0, 4.0]

### 4h. Edge cases - oversized windows and bad arguments

A window larger than the input is a legitimate no-result case: `[]` for the average and rolling percentile, `None` for the max window sum, and the deque simply counts what it has for the event window. Non-positive windows and empty samples raise, because returning a plausible-looking number would hide the caller's bug.

Note the asymmetry: an empty timestamp list is fine and returns `[]`, while an empty *sample* for a percentile raises -- a count of zero events is meaningful, a median of zero values is not.

In [31]:
assert moving_average([1, 2], 3) == []
assert max_window_sum([1, 2], 3) is None
assert max_window_sum([], 1) is None
assert max_window_sum([-5, -1, -3], 2) == -4
assert trailing_event_count([], 3) == []

for bad_call in (
    lambda: moving_average([1, 2], 0),
    lambda: percentile([], 50),
    lambda: percentile([1, 2, 3], 101),
    lambda: rolling_percentile([1, 2], 0, 50),
):
    try:
        bad_call()
    except ValueError:
        pass
    else:
        raise AssertionError("invalid window or percentile must raise ValueError")

### 4i. Stated output check

Re-print the four documented outputs so the cell output in the saved notebook matches the `Example` block exactly. The assert on the trailing counts also checks the left-edge convention: the event at 0 is gone by the time the window at 3 arrives.

In [32]:
print("moving_average            ->", moving_average([1, 2, 3, 4, 5], 3))
print("max_window_sum            ->", max_window_sum([1, 2, 3, 4, 5], 3))
print("percentile(p50)           ->", percentile([10.0, 12.0, 13.0, 15.0, 20.0, 100.0], 50))
print("trailing_event_count      ->", trailing_event_count([0, 1, 2, 10, 11], 3))
assert trailing_event_count([0, 1, 2, 10, 11], 3) == [1, 2, 3, 1, 2]
print("P4 checks passed")

moving_average            -> [2.0, 3.0, 4.0]
max_window_sum            -> 12
percentile(p50)           -> 14.0
trailing_event_count      -> [1, 2, 3, 1, 2]
P4 checks passed


**Complexity & edge cases.** The moving average and event count are O(n) time with O(window) memory (the deque only ever holds live events); `max_window_sum` is O(n) with O(1) extra space; `rolling_percentile` is O(n x w log w) because each window is re-sorted. Edge cases: a window larger than the input yields `[]`/`None`; define whether the window is inclusive at the left edge (here it is exclusive: `t - window` is dropped); empty samples raise in `percentile`; floating-point averages can drift, so use `math.fsum` or Kahan summation for very long windows; p99 on a handful of points is just the maximum and is statistical noise.

**Interview follow-ups.**

1. *How do you compute an exact rolling median efficiently?* Keep a max-heap of the lower half and a min-heap of the upper half, rebalancing on insert/evict: O(log window) per element.
2. *Why are percentiles not additive across buckets?* The p99 of a union is not the max or average of the per-bucket p99s - you must merge the underlying sketches or raw samples.

## Problem 5 - Two pointers and prefix sums

**Statement.** Find two values in a sorted array that sum to a target (two pointers), answer range-sum queries in O(1) (prefix sums), count subarrays summing to a target (prefix sums + hash map), and find the first element at or after a target (`bisect`).

**Example.**

```text
two_sum_sorted([1,3,4,6,8,11], 10)   -> (2, 3)          # values 4 + 6
build_prefix_sums([2,4,6,8])         -> [0, 2, 6, 12, 20]
range_sum(prefix, 1, 3)              -> 10               # 4 + 6
count_subarrays_with_sum([1,1,1], 2) -> 2
first_at_or_after([1,3,3,7], 3)      -> 3
```

**Constraints.** Two pointers is O(n) time and O(1) space on already-sorted input; prefix arrays build in O(n) and answer in O(1); support negative numbers.

### 5a. Two pointers on a sorted array

Start at both ends: if the sum is too small the only way to grow it is to advance the left pointer, and if it is too large the only way to shrink it is to retreat the right pointer. Each index moves at most n times, so the scan is O(n) time and O(1) space.

The precondition is that the data is sorted; on unsorted data the move decision is invalid and you must hash instead. Returning indices (not values) keeps duplicates unambiguous, and any valid pair is acceptable.

In [33]:
from bisect import bisect_left, bisect_right
from collections.abc import Sequence
from itertools import accumulate


def two_sum_sorted(nums: Sequence[int], target: int) -> tuple[int, int] | None:
    """Two-pointer scan on a sorted array; O(n) time, O(1) space."""
    left, right = 0, len(nums) - 1
    while left < right:
        total = nums[left] + nums[right]
        if total == target:
            return left, right
        if total < target:
            left += 1
        else:
            right -= 1
    return None

### 5b. Prefix sums

Store `prefix[i] = sum(values[:i])` with a leading zero, so the array has length `n + 1` and any range `[lo, hi)` is one subtraction: `prefix[hi] - prefix[lo]`. Building is O(n) with `itertools.accumulate`, and every later query is O(1).

The leading zero is what makes `lo = 0` work without a branch. Skipping it is the classic off-by-one that forces a special case for the first element.

In [34]:
def build_prefix_sums(values: Sequence[int]) -> list[int]:
    """prefix[i] = sum(values[:i]); length n + 1 so prefix[0] == 0."""
    return [0, *accumulate(values)]

### 5c. O(1) range queries with validation

With the prefix array in hand, `range_sum` is two lookups. Validate `0 <= lo <= hi <= len(prefix) - 1` up front: an invalid range must raise rather than silently return a negative or wrapped number, because a wrong aggregate is far worse than a failed job.

An empty range (`lo == hi`) is valid and returns 0, matching the mathematical convention.

In [35]:
def range_sum(prefix: Sequence[int], lo: int, hi: int) -> int:
    """sum(values[lo:hi]) in O(1) once the prefix array is built."""
    if not 0 <= lo <= hi <= len(prefix) - 1:
        raise ValueError("invalid range")
    return prefix[hi] - prefix[lo]

### 5d. Counting subarrays that sum to a target

Keep a hash map from each running prefix sum to how many times it has occurred, seeded with `{0: 1}` for the empty prefix. At every step, the number of subarrays ending here is the count of times `running - target` has been seen.

This is O(n) time and O(n) memory, and it is the version that *works with negatives*, where a sliding window does not. The `{0: 1}` seed is what counts subarrays that start at index 0.

In [36]:
def count_subarrays_with_sum(values: Sequence[int], target: int) -> int:
    """Count contiguous subarrays summing to target (works with negatives)."""
    seen = {0: 1}
    running = 0
    count = 0
    for value in values:
        running += value
        count += seen.get(running - target, 0)
        seen[running] = seen.get(running, 0) + 1
    return count

### 5e. Lower and upper bound with `bisect`

`bisect_left` returns the first index at or after the target and `bisect_right - 1` the last index at or before it; both are O(log n) and run in C. Convert the index to a value and return `None` at the edges so callers do not have to catch `IndexError`.

Interviews care about the pair because it answers "first row at or after this watermark" (catch-up) and "last partition at or before this key" (lookup) without scanning. Any monotonic ordering works, which is why ISO date strings are the standard partition key.

In [37]:
def first_at_or_after(sorted_values: Sequence[int], target: int) -> int | None:
    """Lower bound via bisect; None when every value is smaller than target."""
    index = bisect_left(sorted_values, target)
    return sorted_values[index] if index < len(sorted_values) else None


def last_at_or_before(sorted_values: Sequence[int], target: int) -> int | None:
    """Upper bound minus one; None when every value is larger than target."""
    index = bisect_right(sorted_values, target) - 1
    return sorted_values[index] if index >= 0 else None

### 5f. Happy path - two pointers

The documented target 10 is met by the values 4 and 6 at indices 2 and 3. A target no pair can reach returns `None`, and a single-element array cannot contain a pair. Negative inputs are fine because the comparison only depends on the ordering.

In [38]:
assert two_sum_sorted([1, 3, 4, 6, 8, 11], 10) == (2, 3)
assert two_sum_sorted([1, 2, 3], 100) is None
assert two_sum_sorted([1], 1) is None
assert two_sum_sorted([-3, -1, 4, 9], 1) == (0, 2)

### 5g. Happy path - prefix sums and subarray counts

`build_prefix_sums([2, 4, 6, 8])` is `[0, 2, 6, 12, 20]`, so `range_sum(prefix, 1, 3)` is `12 - 2 = 10` (the values 4 and 6) and the full range is 20. The subarray counter finds two subarrays in `[1, 1, 1]` that sum to 2 and four in the alternating `[1, -1, 1, -1]`.

In [39]:
prefix = build_prefix_sums([2, 4, 6, 8])
print("prefix:", prefix, "range[1:3] =", range_sum(prefix, 1, 3))
assert prefix == [0, 2, 6, 12, 20]
assert range_sum(prefix, 1, 3) == 10
assert range_sum(prefix, 0, 4) == 20
assert range_sum(prefix, 2, 2) == 0
assert count_subarrays_with_sum([1, 1, 1], 2) == 2
assert count_subarrays_with_sum([1, -1, 1, -1], 0) == 4

prefix: [0, 2, 6, 12, 20] range[1:3] = 10


### 5h. Happy path - bounds

`first_at_or_after([1, 3, 3, 7], 3)` is the first 3, `last_at_or_before` is the last 3 (duplicates are handled by the two different bisect variants), and a target beyond both ends returns `None`. The empty list must also return `None` instead of raising.

In [40]:
print("first_at_or_after([1,3,3,7], 3) =", first_at_or_after([1, 3, 3, 7], 3))
assert first_at_or_after([1, 3, 3, 7], 3) == 3
assert first_at_or_after([1, 3, 3, 7], 8) is None
assert first_at_or_after([], 1) is None
assert last_at_or_before([1, 3, 3, 7], 3) == 3
assert last_at_or_before([1, 3, 3, 7], 0) is None

first_at_or_after([1,3,3,7], 3) = 3


### 5i. Edge cases - empty inputs and invalid ranges

Empty inputs must be inert everywhere: `count_subarrays_with_sum([], 0)` is 0 because no subarray exists, and the bound helpers return `None`. An inverted range must raise, and the raise happens *before* any indexing so the caller gets a clear error rather than a wrong number.

In [41]:
assert count_subarrays_with_sum([], 0) == 0
assert build_prefix_sums([]) == [0]
assert range_sum(build_prefix_sums([]), 0, 0) == 0

try:
    range_sum(prefix, 3, 1)
except ValueError:
    pass
else:
    raise AssertionError("lo > hi must raise")

### 5j. Stated output check

Print the documented example outputs and assert the two that are not plain numbers. The prefix print makes the leading-zero convention visible, which is exactly what an interviewer will check first.

In [42]:
print("prefix      ->", build_prefix_sums([2, 4, 6, 8]))
print("range[1:3]  ->", range_sum(build_prefix_sums([2, 4, 6, 8]), 1, 3))
print("subarrays   ->", count_subarrays_with_sum([1, 1, 1], 2))
print("first >= 3  ->", first_at_or_after([1, 3, 3, 7], 3))
print("P5 checks passed")

prefix      -> [0, 2, 6, 12, 20]
range[1:3]  -> 10
subarrays   -> 2
first >= 3  -> 3
P5 checks passed


**Complexity & edge cases.** Two pointers is O(n) time / O(1) space but only valid on sorted data; a prefix array builds in O(n) and each query is O(1); `count_subarrays_with_sum` is O(n) time / O(n) memory and handles negatives (sliding windows do not). Edge cases: duplicate values are fine because any valid pair is acceptable; `lo == hi` returns 0; empty input; `bisect_left` gives the first equal element, `bisect_right - 1` gives the last; a monotonic predicate is required for binary search, so date partitions must be zero-padded ISO strings.

**Interview follow-ups.**

1. *Why does a sliding window fail with negative numbers?* Shrinking the window does not monotonically reduce the sum, so you cannot decide when to move the left edge.
2. *How do prefix sums extend to 2-D?* Build a summed-area table where `S[i][j]` is the sum of the rectangle above-left of `(i, j)`; each rectangle query is four lookups.

## Problem 6 - Merge overlapping intervals

**Statement.** Merge every overlapping or touching half-open `[start, end)` interval - how a scheduler coalesces file-availability windows and how compaction groups small files.

**Example.**

```text
merge([[1,3],[2,6],[8,10],[15,18]]) -> [[1,6],[8,10],[15,18]]
merge([[1,4],[4,5]])                -> [[1,5]]     # touching merges
merge([[1,10],[2,3],[4,5]])         -> [[1,10]]    # nested absorbed
```

**Constraints.** O(n log n) time; do not mutate the input; raise on `start > end`; keep zero-length intervals; treat touching endpoints as overlapping (half-open intervals).

### 6a. Sort, then one sweep

Sorting by start time reduces the problem to a single left-to-right pass: each interval either extends the last merged interval (`start <= merged[-1][1]`) or begins a new one. The sort dominates at O(n log n); the sweep is O(n) time and O(n) output.

Treating touching endpoints as overlapping (`<=`) is the half-open convention: `[1, 4)` and `[4, 5)` cover a continuous span with no gap. Switching to `<` gives closed intervals instead -- pick one and document it.

In [43]:
from collections.abc import Iterable


def merge_intervals(intervals: Iterable[tuple[int, int]]) -> list[tuple[int, int]]:
    """Merge overlapping or touching half-open [start, end) intervals."""
    merged: list[tuple[int, int]] = []
    for start, end in sorted(intervals):
        if start > end:
            raise ValueError(f"invalid interval: {(start, end)}")
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    return merged

### 6b. Happy path

The four documented cases: disjoint intervals pass through, an overlap fuses to `(1, 6)`, touching intervals fuse to `(1, 5)`, and a nested interval is absorbed by `(1, 10)`. The `max` in the extension step is what handles nesting -- without it, `[1, 10)` followed by `[2, 3)` would shrink the end to 3.

In [44]:
print(merge_intervals([(1, 3), (2, 6), (8, 10), (15, 18)]))
assert merge_intervals([(1, 3), (2, 6), (8, 10), (15, 18)]) == [(1, 6), (8, 10), (15, 18)]
assert merge_intervals([(1, 4), (4, 5)]) == [(1, 5)]
assert merge_intervals([(1, 10), (2, 3), (4, 5)]) == [(1, 10)]

[(1, 6), (8, 10), (15, 18)]


### 6c. Edge cases - empty, zero-length, unordered, no mutation

`merge_intervals([])` must be `[]`, and a zero-length interval `(5, 5)` is valid and must survive. Because the function sorts a copy, the caller's list order is irrelevant and the caller's list is never reordered -- both properties matter when intervals arrive as a batch from an unpredictable source.

`start > end` is rejected loudly; a silently malformed interval corrupts every later merge.

In [45]:
assert merge_intervals([(5, 5)]) == [(5, 5)]
assert merge_intervals([]) == []

# Input order must not matter, and the input must not be mutated.
data = [(8, 10), (1, 3), (2, 6), (15, 18)]
snapshot = list(data)
assert merge_intervals(data) == [(1, 6), (8, 10), (15, 18)]
assert data == snapshot

try:
    merge_intervals([(3, 1)])
except ValueError:
    pass
else:
    raise AssertionError("start > end must raise")

### 6d. Stated output check

Print the merged list for the documented input and compare with `[[1,6],[8,10],[15,18]]`. `sorted()` returns a list of tuples, so the printed form uses tuple syntax; the asserts are the authoritative check.

In [46]:
print("merged ->", merge_intervals([(1, 3), (2, 6), (8, 10), (15, 18)]))
print("touching ->", merge_intervals([(1, 4), (4, 5)]))
print("nested ->", merge_intervals([(1, 10), (2, 3), (4, 5)]))
print("P6 checks passed")

merged -> [(1, 6), (8, 10), (15, 18)]
touching -> [(1, 5)]
nested -> [(1, 10)]
P6 checks passed


**Complexity & edge cases.** O(n log n) for the sort plus O(n) for the sweep and the output; `sorted` copies, so the input is untouched. Edge cases: empty input returns `[]`; zero-length intervals survive; closed intervals need `start < merged[-1][1]` so touching endpoints stay separate; nested intervals are absorbed by `max`; huge coordinates favour an interval list over a bitmap; streaming intervals that arrive unordered require a buffer and can only be partially merged.

**Interview follow-ups.**

1. *How would you find the peak number of concurrent intervals?* Sweep the endpoints as `(+1, start)` and `(-1, end)` events, sort, and track a running maximum - O(n log n), or O(n) counting sort on small integer ranges.
2. *How does this relate to small-file compaction?* Group adjacent files whose size range overlaps or is below a threshold, then rewrite each group as one file - the same merge-with-tolerance idea.

## Problem 7 - Binary search over sorted partitions

**Statement.** A table is stored as sorted ISO-date partitions. (a) Find the partition that can contain a key, or the first partition at or after a watermark. (b) Generalise the partitioning idea: find the k-th smallest element of two sorted arrays by binary-searching the split point, which is the "median of two sorted arrays" interview classic.

**Example.**

```text
partitions = ["2026-01-01","2026-02-01","2026-03-01","2026-04-01"]
partition_for_key("2026-03-15")   -> "2026-03-01"
first_at_or_after("2026-03-15")   -> "2026-04-01"

kth_smallest_two_sorted([1,3,8], [2,4,5,6,7], k=4) -> 4
```

**Constraints.** O(log n) per lookup; O(log(min(n, m))) for the two-array selection; handle empty inputs and out-of-range `k`; the sorted arrays must be ascending (raise if not).

### 7a. Partition lookup for a key

`bisect_right(partitions, key) - 1` is the last partition start that is `<=` the key: the file that could contain it. This is the prune step in partition elimination, and it is O(log n) with no scanning.

`bisect_right` (not `bisect_left`) is what makes an exact partition-boundary key resolve to its own partition rather than the previous one. A key before every partition returns `None`.

In [47]:
from bisect import bisect_left, bisect_right
from collections.abc import Sequence


def partition_for_key(partitions: Sequence[str], key: str) -> str | None:
    """Last partition whose start is <= key (the file that can contain key)."""
    index = bisect_right(partitions, key) - 1
    return partitions[index] if index >= 0 else None

### 7b. Watermark resume point

`bisect_left(partitions, watermark)` is the first partition at or after the watermark, i.e. where incremental processing resumes after the last committed watermark. The two helpers are deliberately mirrored: one looks back for a lookup, one looks forward for catch-up.

The partition strings must be zero-padded ISO dates, because `bisect` compares lexicographically and `"2026-1-01"` would sort after `"2026-12-01"`.

In [48]:
def first_at_or_after(partitions: Sequence[str], watermark: str) -> str | None:
    """First partition at or after a watermark (where incremental processing resumes)."""
    index = bisect_left(partitions, watermark)
    return partitions[index] if index < len(partitions) else None

### 7c. k-th smallest of two sorted arrays

Binary-search the *split point* instead of the value: take `i` elements from `a` and `j = k - i` from `b`, and check that the largest left element is no bigger than the smallest right element on both sides. Each step halves the search range, so the selection is O(log(min(n, m))).

Always search the shorter array: it bounds the range and keeps both partition indices inside their arrays. The sentinels `-inf`/`inf` remove the boundary special cases. Unsorted input must be rejected explicitly, because the invariant fails silently otherwise.

In [49]:
def kth_smallest_two_sorted(a: Sequence[int], b: Sequence[int], k: int) -> int:
    """k-th smallest (1-indexed) of two sorted arrays via partition binary search."""
    if not 1 <= k <= len(a) + len(b):
        raise ValueError("k out of range")
    if any(a[i] > a[i + 1] for i in range(len(a) - 1)) or any(b[i] > b[i + 1] for i in range(len(b) - 1)):
        raise ValueError("inputs must be sorted")
    if len(a) > len(b):
        a, b = b, a  # binary-search the shorter array
    n, m = len(a), len(b)
    low, high = max(0, k - m), min(k, n)
    while low <= high:
        i = (low + high) // 2          # take i elements from a
        j = k - i                      # ... and j elements from b
        a_left = a[i - 1] if i > 0 else float("-inf")
        a_right = a[i] if i < n else float("inf")
        b_left = b[j - 1] if j > 0 else float("-inf")
        b_right = b[j] if j < m else float("inf")
        if a_left <= b_right and b_left <= a_right:
            return max(a_left, b_left)
        if a_left > b_right:
            high = i - 1
        else:
            low = i + 1
    raise RuntimeError("unreachable: inputs were not sorted")

### 7d. Median of two sorted arrays

An even total length needs the average of the two middle order statistics, an odd total just the middle one. Building `median_two_sorted` on top of the k-th selection keeps one tested primitive and makes the even/odd branch obvious.

Two empty arrays have no median, so raise; one empty array is fine and reduces to a normal index.

In [50]:
def median_two_sorted(a: Sequence[int], b: Sequence[int]) -> float:
    """Median of two sorted arrays built on the k-th selection."""
    total = len(a) + len(b)
    if total == 0:
        raise ValueError("both arrays are empty")
    if total % 2 == 1:
        return float(kth_smallest_two_sorted(a, b, total // 2 + 1))
    lower = kth_smallest_two_sorted(a, b, total // 2)
    upper = kth_smallest_two_sorted(a, b, total // 2 + 1)
    return (lower + upper) / 2

### 7e. Happy path - partition lookups

For the documented partitions, the key `2026-03-15` falls inside the March partition, an exact boundary key resolves to its own partition, and a key before the first partition returns `None`. The watermark helpers mirror that: a mid-month watermark resumes at the next partition, an exact boundary resumes at itself, and a watermark past the end returns `None`.

In [51]:
partitions = ["2026-01-01", "2026-02-01", "2026-03-01", "2026-04-01"]
print("partition_for_key:", partition_for_key(partitions, "2026-03-15"))
assert partition_for_key(partitions, "2026-03-15") == "2026-03-01"
assert partition_for_key(partitions, "2026-01-01") == "2026-01-01"
assert partition_for_key(partitions, "2025-12-31") is None
assert partition_for_key([], "2026-01-01") is None
assert first_at_or_after(partitions, "2026-03-15") == "2026-04-01"
assert first_at_or_after(partitions, "2026-03-01") == "2026-03-01"
assert first_at_or_after(partitions, "2027-01-01") is None

partition_for_key: 2026-03-01


### 7f. Happy path - every order statistic

Cross-check the selection against the obvious (slow) reference: `sorted(a + b)` and index `k - 1`. Looping over every valid `k` is the strongest cheap test of a partition algorithm, because an off-by-one in `i` or `j` usually shows up for only one value of `k`.

One empty array and the two median cases (odd and even total length) round out the happy path.

In [52]:
a, b = [1, 3, 8], [2, 4, 5, 6, 7]
merged = sorted(a + b)
print("merged:", merged)
for k in range(1, len(merged) + 1):
    assert kth_smallest_two_sorted(a, b, k) == merged[k - 1]
assert kth_smallest_two_sorted([], [1, 2, 3], 2) == 2
assert kth_smallest_two_sorted([1, 2], [], 1) == 1
assert median_two_sorted([1, 3], [2]) == 2.0
assert median_two_sorted([1, 2], [3, 4]) == 2.5

merged: [1, 2, 3, 4, 5, 6, 7, 8]


### 7g. Edge cases - invalid input must raise

Three invalid calls: `k` larger than the combined length, an unsorted array, and two empty arrays for the median. All three raise `ValueError` rather than returning a plausible number, because a silently wrong order statistic is much harder to debug than an exception at the boundary.

In [53]:
for bad_call in (
    lambda: kth_smallest_two_sorted([1], [2], 3),
    lambda: kth_smallest_two_sorted([2, 1], [3], 2),
    lambda: median_two_sorted([], []),
):
    try:
        bad_call()
    except ValueError:
        pass
    else:
        raise AssertionError("invalid input must raise ValueError")

### 7h. Stated output check

Print the two documented lookups and the merged reference array. The assert re-checks the k-th value for the documented `k = 4`, which should be 4.

In [54]:
print("partition_for_key(2026-03-15) ->", partition_for_key(partitions, "2026-03-15"))
print("first_at_or_after(2026-03-15) ->", first_at_or_after(partitions, "2026-03-15"))
print("kth_smallest_two_sorted(k=4)  ->", kth_smallest_two_sorted([1, 3, 8], [2, 4, 5, 6, 7], 4))
assert kth_smallest_two_sorted([1, 3, 8], [2, 4, 5, 6, 7], 4) == 4
print("P7 checks passed")

partition_for_key(2026-03-15) -> 2026-03-01
first_at_or_after(2026-03-15) -> 2026-04-01
kth_smallest_two_sorted(k=4)  -> 4
P7 checks passed


**Complexity & edge cases.** `bisect` gives O(log n) lookups in C; the two-array selection is O(log(min(n, m))) time and O(1) space. Edge cases: empty partition lists; duplicate partition starts (`bisect_left` -> first, `bisect_right` -> last); lexicographic order only equals chronological order for zero-padded ISO dates; `k` out of range; one empty array; even total length needs the average of two order statistics; unsorted input must be rejected, because the partition invariant silently breaks.

**Interview follow-ups.**

1. *Why binary-search the shorter array?* It bounds the search range at min(n, m) and keeps the partition indices valid, giving the tighter O(log(min(n,m))) bound.
2. *How do you binary-search data in a data lake?* Store per-file min/max statistics (Parquet footers, Iceberg/Delta manifests) and binary-search the sorted manifest instead of scanning every file.

## Problem 8 - Generator-based streaming of a large file

**Statement.** Process a file far larger than RAM on one machine. Read it lazily, transform in bounded batches, and never hold the whole file. Generators keep memory flat.

**Example.**

```text
file   : events.csv.gz with lines "user_id,amount" (millions of rows)
target : total amount per user, O(distinct users) memory
search : first N matching lines, short-circuiting the read
```

**Constraints.** O(1) memory for the reader, independent of file size; handle plain and gzip files; skip blank lines; dead-letter malformed rows instead of aborting; no third-party libraries.

### 8a. A lazy line reader

A generator function holds one line at a time and suspends between yields, so the open file handle is the only state. Iterating a file object already streams with a bounded buffer -- the generator just layers CRLF stripping and transparent gzip opening on top.

The `with` block closes the handle when the generator is exhausted *or* garbage-collected early. Never call `read()`/`readlines()` here; that is the single decision that makes the difference between working and OOM on a file bigger than RAM.

In [55]:
import gzip
from collections import defaultdict
from collections.abc import Iterator
from pathlib import Path

try:  # Python 3.12+
    from itertools import batched
except ImportError:  # Python 3.11 and older, including some Colab runtimes
    from itertools import islice

    def batched(iterable, n):
        "Batch data into tuples of length n; the last batch may be shorter."
        it = iter(iterable)
        while chunk := tuple(islice(it, n)):
            yield chunk


def read_lines(path: Path) -> Iterator[str]:
    """Lazily yield stripped lines, transparently handling .gz files."""
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as handle:
        for line in handle:
            yield line.rstrip("\r\n")

### 8b. One streaming pass to per-user totals

Consume the lazily-read lines in fixed batches and accumulate into a dict keyed by user. Memory is O(batch size + number of distinct users), not O(file size) -- this is the standard single-pass aggregation that a map-reduce reducer performs per partition.

Malformed rows go to the dead-letter path: skip and continue rather than aborting the whole job. Note the order of operations: cast first, *then* add, because `totals[user_id] += value` on a `defaultdict` would insert `0.0` before the cast raises and leave a ghost user behind.

In [56]:
def totals_by_user(path: Path) -> dict[str, float]:
    """One streaming pass: O(batch + distinct users) memory, not O(file)."""
    totals: dict[str, float] = defaultdict(float)
    for batch in batched(read_lines(path), 1_000):
        for line in batch:
            if not line:
                continue
            user_id, _, amount = line.partition(",")
            try:
                value = float(amount)
            except ValueError:
                continue  # dead-letter malformed rows instead of aborting
            # Add only after the cast succeeds: `totals[user_id] += ...` would
            # create a 0.0 entry via defaultdict before the cast raises.
            totals[user_id] += value
    return dict(totals)

### 8c. Short-circuiting search

`first_n_matching` stops reading as soon as it has `n` hits, which is the whole benefit of a generator: an early `break` closes the underlying file instead of scanning the remaining gigabytes. This is how a "find the first bad record in today's dump" tool stays fast.

The alternative -- filtering a materialised list -- pays for the full scan before returning anything.

In [57]:
def first_n_matching(path: Path, needle: str, n: int) -> list[str]:
    """Short-circuiting search over a huge file: stop as soon as n matches exist."""
    matches: list[str] = []
    for line in read_lines(path):
        if needle in line:
            matches.append(line)
            if len(matches) == n:
                break
    return matches

### 8d. Happy path - plain and gzip files

Write a small fixture with a quoted-free CSV, a blank line, and one malformed row, then read it back. The malformed row is skipped, the blank line is skipped, and `u1` accumulates 10 + 2.5. The same function reads the `.gz` fixture without a code change because `read_lines` picks the opener from the suffix.

In [58]:
import gzip
import itertools
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    tmp_path = Path(tmp)
    raw = tmp_path / "events.csv"
    raw.write_text(
        "u1,10\n\n" + "u2,5\n" + "u1,2.5\n" + "bad,not-a-number\n" + "u3,7",
        encoding="utf-8",
    )
    print("totals:", totals_by_user(raw))
    assert totals_by_user(raw) == {"u1": 12.5, "u2": 5.0, "u3": 7.0}
    assert first_n_matching(raw, "u1", 1) == ["u1,10"]
    assert first_n_matching(raw, "missing", 5) == []

    gz = tmp_path / "events.csv.gz"
    with gzip.open(gz, "wt", encoding="utf-8") as handle:
        handle.write("a,1\nb,2\n")
    assert totals_by_user(gz) == {"a": 1.0, "b": 2.0}

totals: {'u1': 12.5, 'u2': 5.0, 'u3': 7.0}


### 8e. Edge cases - laziness and bounded state

A generator's body does not run until the first `next()`, so an infinite source is harmless as long as the consumer stops early. The `explode` probe proves it: the only way to observe the `AssertionError` is to advance the generator, and `close()` disposes of it without finishing.

This is also the test that a reader is truly lazy: if `read_lines` eagerly read the file, the infinite example could not be constructed at all.

In [59]:
def naturals():
    value = 0
    while True:
        yield value
        value += 1


# Generators are lazy: an infinite source is fine as long as you stop early.
assert list(itertools.islice(naturals(), 3)) == [0, 1, 2]
gen = naturals()
assert next(gen) == 0 and next(gen) == 1  # state advances only on demand


def explode():
    yield 1
    raise AssertionError("generator body must not run eagerly")


probe = explode()
assert next(probe) == 1  # nothing raises until the generator is advanced
probe.close()

### 8f. Stated output check

Re-create a tiny fixture here and print the per-user totals: `u1` should accumulate `10 + 2.5 = 12.5` while the malformed `bad` row is dead-lettered. The generator-size assertion then shows that a source of a billion rows still holds constant state.

In [60]:
import sys
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    sample = Path(tmp) / "events.csv"
    sample.write_text("u1,10\nu1,2.5\nbad,not-a-number\n", encoding="utf-8")
    print("totals ->", totals_by_user(sample))
    assert totals_by_user(sample) == {"u1": 12.5}

large_generator = (line for line in range(10**9))
assert sys.getsizeof(large_generator) < sys.getsizeof([0] * 10_000)
next(large_generator)               # state advances only on demand
large_generator.close()
print("P8 checks passed")

totals -> {'u1': 12.5}
P8 checks passed


**Complexity & edge cases.** O(n) time, O(batch size + retained state) memory; the reader holds one line at a time. Edge cases: a final line without a newline; CRLF endings; malformed records should be dead-lettered with a reason rather than killing the job; sniff gzip magic bytes instead of trusting the suffix in production; parallel workers need byte ranges that resync to the next newline; closing a generator early still runs the `with` block via `GeneratorExit`.

**Interview follow-ups.**

1. *How do you parallelise a single huge file?* Give each worker a byte range, seek, discard the partial first line, and stop at the last newline; gzip blocks can be decompressed in parallel with `zlib`-level offsets.
2. *Generator vs `itertools.batched` list chunks - what changes?* Batching gives amortised per-item overhead and lets you hand whole chunks to a vectorised or bulk-insert path, at the cost of holding one batch in memory.

## Problem 9 - Parse CSV, JSON, and NDJSON

**Statement.** Read the common landing formats without pandas: header CSV (with quoted fields and a BOM), a JSON array, NDJSON (one object per line), and a memory-flat streaming JSON array via `JSONDecoder.raw_decode`.

**Example.**

```text
orders.csv   : order_id,amount,note  /  1,10.5,"a, b"
orders.json  : [{"order_id": 1, "amount": 10.5}]
orders.ndjson: {"order_id": 1}\n{"order_id": 2}
```

**Constraints.** O(1) memory for CSV/NDJSON; report bad NDJSON with a line number; reject a non-array JSON document in `parse_json_array`; strip a UTF-8 BOM; no pandas.

### 9a. Header CSV with `csv.DictReader`

`csv.DictReader` maps each row onto the header names and understands quoting, so `"a, b"` stays one field. Wrapping the text in `io.StringIO` keeps the function testable on a string while the same code works on an open file object.

The BOM is stripped explicitly: Excel and many exporters prepend `\ufeff`, which otherwise glues itself to the first header name and breaks every lookup on that column. In production you would open with `encoding="utf-8-sig"` rather than slicing.

In [61]:
import csv
import io
import json
from collections.abc import Iterator
from typing import Any


def parse_csv(text: str) -> list[dict[str, str]]:
    """Header-based CSV; csv handles quoted fields that contain commas/newlines."""
    return list(csv.DictReader(io.StringIO(text.lstrip("\ufeff"))))

### 9b. A JSON array

`json.loads` returns a Python object, but the caller expects a list of records, so a top-level object or scalar is a contract violation and must raise. `parse_json_array` therefore materialises the whole document -- acceptable for a small config payload, not for a multi-GB export.

Deeply nested documents are where `json` recursion limits and memory both bite; that is the cue to reach for streaming.

In [62]:
def parse_json_array(text: str) -> list[dict[str, Any]]:
    """A JSON array must be materialised unless you stream it with raw_decode."""
    payload = json.loads(text)
    if not isinstance(payload, list):
        raise ValueError("expected a JSON array")
    return payload

### 9c. NDJSON with line numbers

One JSON document per line is the natural landing format for streams: each line is independent, so a writer can append and a reader can parallelise or resume. Blank lines are skipped, and a malformed line is re-raised as a `ValueError` carrying its 1-based line number.

The line number is the whole point -- "bad NDJSON somewhere in 4 GB" is not actionable, "bad NDJSON at line 91,233" is.

In [63]:
def parse_ndjson(text: str) -> list[dict[str, Any]]:
    """One JSON object per line; blank lines skipped, bad lines reported with a line number."""
    records: list[dict[str, Any]] = []
    for number, line in enumerate(text.splitlines(), start=1):
        if not line.strip():
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as exc:
            raise ValueError(f"bad NDJSON at line {number}") from exc
    return records

### 9d. Streaming a JSON array with `raw_decode`

`JSONDecoder.raw_decode(text, index)` parses exactly one value and returns the index where it stopped, so you can walk a large buffer one element at a time instead of building the whole list. The loop skips the structural characters that sit between values (`[`, `]`, `,`, whitespace) and yields each decoded value.

Real code feeds the decoder from a chunked file reader and keeps a tail buffer; the point of the exercise is that the parser hands you a resumable position, which is what makes streaming possible at all.

In [64]:
def iter_json_array(text: str) -> Iterator[Any]:
    """Stream a JSON array value-by-value with raw_decode, so it need not fit in memory."""
    decoder = json.JSONDecoder()
    position = 0
    length = len(text)
    while position < length:
        while position < length and text[position] in " \t\r\n,[]":
            position += 1
        if position >= length:
            break
        value, position = decoder.raw_decode(text, position)
        yield value

### 9e. Happy path - CSV and JSON

The CSV fixture has a quoted field containing a comma (`"a, b"`) and an empty trailing field, both of which `csv` handles correctly. The BOM case is a separate assertion because it is invisible in the output until a column lookup fails. The JSON array parses back to the documented records.

In [65]:
csv_text = 'order_id,amount,note\n1,10.5,"a, b"\n2,3.0,\n'
print("csv:", parse_csv(csv_text))
assert parse_csv(csv_text) == [
    {"order_id": "1", "amount": "10.5", "note": "a, b"},
    {"order_id": "2", "amount": "3.0", "note": ""},
]
assert parse_csv("\ufeffid,v\n1,2\n") == [{"id": "1", "v": "2"}]  # BOM stripped

json_text = '[{"order_id": 1, "amount": 10.5}, {"order_id": 2, "amount": 3.0}]'
assert parse_json_array(json_text) == [
    {"order_id": 1, "amount": 10.5},
    {"order_id": 2, "amount": 3.0},
]

csv: [{'order_id': '1', 'amount': '10.5', 'note': 'a, b'}, {'order_id': '2', 'amount': '3.0', 'note': ''}]


### 9f. Happy path - NDJSON and the streaming decoder

NDJSON skips blank lines and returns the objects in order. The streaming decoder walks a deliberately ugly buffer (`"  [ {...}, {...}, 3 ] "`) to prove it skips commas, brackets, and whitespace, not just newlines; `"[]"` yields nothing.

In [66]:
ndjson_text = '{"order_id": 1}\n\n{"order_id": 2}\n'
assert parse_ndjson(ndjson_text) == [{"order_id": 1}, {"order_id": 2}]
assert parse_ndjson("") == []

streamed = list(iter_json_array('  [ {"a": 1}, {"b": [1, 2]}, 3 ] '))
print("streamed JSON:", streamed)
assert streamed == [{"a": 1}, {"b": [1, 2]}, 3]
assert list(iter_json_array("[]")) == []

streamed JSON: [{'a': 1}, {'b': [1, 2]}, 3]


### 9g. Edge cases - non-array JSON and bad NDJSON

A top-level JSON object is a schema-contract failure, and a malformed NDJSON line must name its line. Both raise `ValueError`, which the caller can route to a dead-letter path with the raw payload attached.

Empty NDJSON is valid and returns `[]` -- an empty file is not an error, it is a zero-row batch.

In [67]:
try:
    parse_json_array('{"order_id": 1}')
except ValueError:
    pass
else:
    raise AssertionError("non-array JSON must raise")

try:
    parse_ndjson('{"ok": 1}\nnot json\n')
except ValueError as exc:
    assert "line 2" in str(exc)
else:
    raise AssertionError("bad NDJSON must raise")

### 9h. Stated output check

Print the parsed CSV rows and the streamed JSON values. The CSV output shows the quoted comma preserved as a single field and the missing note as an empty string, which is exactly how a downstream loader should see it.

In [68]:
print("csv      ->", parse_csv(csv_text))
print("ndjson   ->", parse_ndjson(ndjson_text))
print("streamed ->", list(iter_json_array('  [ {"a": 1}, {"b": [1, 2]}, 3 ] ')))
print("P9 checks passed")

csv      -> [{'order_id': '1', 'amount': '10.5', 'note': 'a, b'}, {'order_id': '2', 'amount': '3.0', 'note': ''}]
ndjson   -> [{'order_id': 1}, {'order_id': 2}]
streamed -> [{'a': 1}, {'b': [1, 2]}, 3]
P9 checks passed


**Complexity & edge cases.** CSV and NDJSON are O(n) time / O(1) memory; a JSON array is O(n) memory because the document must exist. `raw_decode` makes the array streamable but requires you to skip separators yourself. Edge cases: a UTF-8 BOM (`utf-8-sig`); embedded newlines inside quoted CSV fields; very long NDJSON lines; empty strings that break `float`; a JSON file that is an object rather than an array; deeply nested JSON needs `ijson` for true streaming.

**Interview follow-ups.**

1. *Why is NDJSON the preferred landing format for streams?* Each line is an independent, appendable, partially readable document, so a crash mid-file loses at most one line and readers can parallelise by line ranges.
2. *How do you make parsing robust to schema drift?* Parse into `dict`, coerce types with an explicit per-column cast function, and route cast failures to a dead-letter path with the raw line and error.

## Problem 10 - Flatten nested JSON into rows

**Statement.** Turn nested documents into tabular rows: (a) `flatten_record` collapses nested objects into dotted paths and indexes arrays; (b) `explode_rows` emits one output row per array element with the parent fields repeated - the in-memory equivalent of `UNNEST` / `LATERAL VIEW explode`.

**Example.**

```text
input : {"order_id":1,"customer":{"country":"us"},
         "items":[{"sku":"a","qty":2},{"sku":"b","qty":1}]}
flat  : {"order_id":1,"customer.country":"us",
         "items[0].sku":"a","items[0].qty":2,"items[1].sku":"b","items[1].qty":1}
rows  : [{"order_id":1,"customer":{...},"sku":"a","qty":2},
         {"order_id":1,"customer":{...},"sku":"b","qty":1}]
```

**Constraints.** Preserve key order; do not mutate the input; an empty array emits one parent row (left-join semantics); a missing array field behaves like an empty array.

### 10a. Recursive flatten with dotted paths

Walk the document depth-first and build a path string as you descend; dict keys extend the path with `sep`, list elements extend it with `[index]`, and scalars are written to the output. Insertion order is preserved because `out` is filled in traversal order, which is what lets a column-selecting reader stay stable.

A literal key that already contains the separator collides with a nested path -- production code escapes it. Deeply nested input can also hit the recursion limit, at which point an explicit stack is the safer rewrite.

In [69]:
from collections.abc import Iterable
from typing import Any


def flatten_record(record: dict[str, Any], prefix: str = "", sep: str = ".") -> dict[str, Any]:
    """Collapse nested objects into dotted paths; arrays are indexed."""
    out: dict[str, Any] = {}
    for key, value in record.items():
        path = f"{prefix}{sep}{key}" if prefix else key
        if isinstance(value, dict):
            out.update(flatten_record(value, path, sep))
        elif isinstance(value, list):
            for index, element in enumerate(value):
                if isinstance(element, dict):
                    out.update(flatten_record(element, f"{path}[{index}]", sep))
                else:
                    out[f"{path}[{index}]"] = element
        else:
            out[path] = value
    return out

### 10b. Explode - one row per array element

For each parent record, copy every field except the array, then emit one row per element; when the element is a dict its keys are merged into the row, so `{"sku": ..., "qty": ...}` becomes columns. The parent fields are *repeated* on every generated row, which is exactly the row-multiplication that `UNNEST` produces.

An empty or missing array emits a single parent row, matching a left join. That choice is what keeps the parent count conserved; dropping the parent would silently lose orders with no items.

In [70]:
def explode_rows(records: Iterable[dict[str, Any]], array_field: str) -> list[dict[str, Any]]:
    """One output row per array element, repeating the parent fields (UNNEST)."""
    rows: list[dict[str, Any]] = []
    for record in records:
        values = record.get(array_field) or []
        base = {key: value for key, value in record.items() if key != array_field}
        if not values:
            rows.append(dict(base))  # left-join semantics: keep one row
            continue
        for value in values:
            row = dict(base)
            if isinstance(value, dict):
                row.update(value)
            else:
                row[array_field] = value
            rows.append(row)
    return rows

### 10c. Happy path - flatten

The documented order flattens to nine scalar columns: the top-level `order_id`, two dotted `customer.*` paths, four indexed `items[*].*` paths, and two indexed `tags[*]` values. The exact dict comparison checks both the values and the traversal order.

In [71]:
order = {
    "order_id": 1,
    "customer": {"country": "us", "city": "austin"},
    "items": [{"sku": "a", "qty": 2}, {"sku": "b", "qty": 1}],
    "tags": ["gift", "rush"],
}
flat = flatten_record(order)
print("flat:", flat)
assert flat == {
    "order_id": 1,
    "customer.country": "us",
    "customer.city": "austin",
    "items[0].sku": "a",
    "items[0].qty": 2,
    "items[1].sku": "b",
    "items[1].qty": 1,
    "tags[0]": "gift",
    "tags[1]": "rush",
}

flat: {'order_id': 1, 'customer.country': 'us', 'customer.city': 'austin', 'items[0].sku': 'a', 'items[0].qty': 2, 'items[1].sku': 'b', 'items[1].qty': 1, 'tags[0]': 'gift', 'tags[1]': 'rush'}


### 10d. Happy path - explode

The two-item order produces two rows, each carrying `order_id`, the nested `customer`, and the full `tags` list, plus the item's own `sku` and `qty`. The input document is asserted unchanged afterwards, because the function builds new dicts rather than editing in place.

In [72]:
rows = explode_rows([order], "items")
print("exploded rows:", len(rows))
assert rows == [
    {
        "order_id": 1,
        "customer": {"country": "us", "city": "austin"},
        "tags": ["gift", "rush"],
        "sku": "a",
        "qty": 2,
    },
    {
        "order_id": 1,
        "customer": {"country": "us", "city": "austin"},
        "tags": ["gift", "rush"],
        "sku": "b",
        "qty": 1,
    },
]
assert order["items"] == [{"sku": "a", "qty": 2}, {"sku": "b", "qty": 1}]  # not mutated

exploded rows: 2


### 10e. Edge cases - empty arrays, missing arrays, scalars, empty input

An empty array and a missing array both emit the parent row minus the array field, which is the left-join contract. An array of scalars keeps the original field name and repeats it once per value. `flatten_record({})` is `{}`, and `explode_rows([], ...)` is `[]`.

In [73]:
assert flatten_record({}) == {}
assert flatten_record({"a": {"b": {"c": 1}}}) == {"a.b.c": 1}
assert flatten_record({"a": 1}, sep="/") == {"a": 1}
assert explode_rows([{"id": 1, "items": []}], "items") == [{"id": 1}]
assert explode_rows([{"id": 1}], "items") == [{"id": 1}]
assert explode_rows([{"id": 1, "items": ["x", "y"]}], "items") == [
    {"id": 1, "items": "x"},
    {"id": 1, "items": "y"},
]
assert explode_rows([], "items") == []

### 10f. Stated output check

Print the flattened keys in order and the exploded rows so the saved output matches the `Example` block. The number of rows equals the total number of item elements across all parents, which is the count a downstream aggregate must expect.

In [74]:
print("flat keys ->", list(flat))
print("rows      ->", [(r["sku"], r["qty"]) for r in rows])
print("P10 checks passed")

flat keys -> ['order_id', 'customer.country', 'customer.city', 'items[0].sku', 'items[0].qty', 'items[1].sku', 'items[1].qty', 'tags[0]', 'tags[1]']
rows      -> [('a', 2), ('b', 1)]
P10 checks passed


**Complexity & edge cases.** Both functions are O(total nodes) time; `flatten_record` uses O(depth) recursion and `explode_rows` materialises one row per array element. Edge cases: a missing parent key; an empty array should still emit one parent row to match a left join; arrays of arrays and null array elements; a literal key containing the separator collides with a nested path, so production code should escape it; deeply nested documents can exceed the recursion limit, so an explicit stack is safer.

**Interview follow-ups.**

1. *Flatten or explode - how do you choose?* Flatten when you need a wide, one-row-per-document table; explode when downstream aggregation needs one row per child event, accepting the row multiplication.
2. *How does row multiplication affect downstream counts?* A naive `COUNT(*)` after exploding counts child records, not orders - count distinct parent ids or pre-aggregate before the explode.

## Problem 11 - In-memory hash join of two datasets

**Statement.** Join a large fact stream to a small dimension without a database. Build a hash map on the smaller side and stream the larger side through it - the same shape as a broadcast join. Support inner and left joins, duplicate build keys, and SQL null semantics.

**Example.**

```text
facts      : [("c1","o1"), ("c2","o2"), ("c9","o3")]
dimensions : [("c1","Alice"), ("c2","Bob")]
inner      : [("c1","o1","Alice"), ("c2","o2","Bob")]
left       : [..., ("c9","o3",None)]
```

**Constraints.** O(n + m) time, O(m) memory for the build side; always build on the smaller side; duplicate build keys multiply rows; NULL keys never match; reject unknown join types.

### 11a. Materialised inner/left hash join

Index the dimension rows by key first (a `defaultdict(list)` so duplicate keys are kept), then probe with each fact row. Hits are emitted once per matching dimension row; misses are emitted with a `None` payload only for a left join.

Two SQL semantics are encoded deliberately: a `None` build key is skipped, and a `None` probe key never matches, because `NULL = NULL` is unknown in SQL. Getting either wrong changes the row count silently.

In [75]:
from collections import defaultdict
from collections.abc import Iterable, Iterator
from typing import Any, TypeVar

K = TypeVar("K")
V = TypeVar("V")


def hash_join(
    facts: Iterable[tuple[K, V]],
    dimensions: Iterable[tuple[K, Any]],
    how: str = "inner",
    null: Any = None,
) -> list[tuple[K, V, Any]]:
    """Broadcast-style join: build a hash index on the right (small) side."""
    if how not in {"inner", "left"}:
        raise ValueError("how must be 'inner' or 'left'")
    index: dict[K, list[Any]] = defaultdict(list)
    for key, value in dimensions:
        if key is not None:  # SQL semantics: NULL keys never match
            index[key].append(value)
    out: list[tuple[K, V, Any]] = []
    for key, fact_value in facts:
        matches = index.get(key) if key is not None else None
        if matches:
            for dim_value in matches:
                out.append((key, fact_value, dim_value))
        elif how == "left":
            out.append((key, fact_value, null))
    return out

### 11b. The same join as a generator

Yielding instead of appending keeps the fact side un-materialised: memory is the build index only, which is the actual broadcast-join property. The output is consumed once, so the caller can write it straight to a sink or feed another stage.

If the consumer needs to iterate twice, it must call the function again or wrap the result in a list -- the classic generator trap.

In [76]:
def hash_join_streaming(
    facts: Iterable[tuple[K, V]],
    dimensions: Iterable[tuple[K, Any]],
    how: str = "inner",
) -> Iterator[tuple[K, V, Any]]:
    """Same join as a generator, so the fact side is never materialised."""
    index: dict[K, list[Any]] = defaultdict(list)
    for key, value in dimensions:
        if key is not None:
            index[key].append(value)
    for key, fact_value in facts:
        matches = index.get(key) if key is not None else None
        if matches:
            for dim_value in matches:
                yield key, fact_value, dim_value
        elif how == "left":
            yield key, fact_value, None

### 11c. Happy path - inner and left

The documented join produces two inner rows and, for the left variant, the unmatched fact `("c9", "o3")` with a `None` payload. The streaming generator must produce the identical inner sequence.

In [77]:
facts = [("c1", "o1"), ("c2", "o2"), ("c9", "o3")]
dims = [("c1", "Alice"), ("c2", "Bob")]
print("inner:", hash_join(facts, dims))
assert hash_join(facts, dims) == [("c1", "o1", "Alice"), ("c2", "o2", "Bob")]
assert hash_join(facts, dims, how="left") == [
    ("c1", "o1", "Alice"),
    ("c2", "o2", "Bob"),
    ("c9", "o3", None),
]
assert list(hash_join_streaming(facts, dims)) == [("c1", "o1", "Alice"), ("c2", "o2", "Bob")]

inner: [('c1', 'o1', 'Alice'), ('c2', 'o2', 'Bob')]


### 11d. Happy path - duplicate build keys multiply rows

One fact row matching two dimension rows produces two output rows, exactly like a SQL join on a non-unique key. This is correct semantics but a common source of exploded output counts, so it deserves an explicit test rather than a comment.

In [78]:
assert hash_join([("c1", "o1")], [("c1", "A"), ("c1", "B")]) == [
    ("c1", "o1", "A"),
    ("c1", "o1", "B"),
]

### 11e. Edge cases - NULL keys, empty sides, unknown join type

A `None` key never matches, so the inner join is empty; the left join still emits the fact row with a null payload so no fact is lost. Empty facts give no rows, empty dimensions make every inner join empty, and an unsupported join type is rejected up front rather than silently behaving like an inner join.

In [79]:
assert hash_join([(None, "o1")], [(None, "A")]) == []
assert hash_join([(None, "o1")], [(None, "A")], how="left") == [(None, "o1", None)]
assert hash_join([], dims) == []
assert hash_join(facts, []) == []

try:
    hash_join([], [], how="outer")
except ValueError:
    pass
else:
    raise AssertionError("unsupported join type must raise")

### 11f. Stated output check

Print the inner and left results for the documented inputs. Compare the left result with the example: the third tuple must be `('c9', 'o3', None)`, which is the anti-join row a left join preserves.

In [80]:
print("inner ->", hash_join(facts, dims))
print("left  ->", hash_join(facts, dims, how="left"))
print("P11 checks passed")

inner -> [('c1', 'o1', 'Alice'), ('c2', 'o2', 'Bob')]
left  -> [('c1', 'o1', 'Alice'), ('c2', 'o2', 'Bob'), ('c9', 'o3', None)]
P11 checks passed


**Complexity & edge cases.** O(n + m) time and O(m) memory for the m-row build side; building on the large side silently materialises the fact table. Duplicate build keys multiply rows, which is correct SQL semantics but can explode output. Edge cases: NULL keys never match in SQL (unlike `==` in Python); `1` vs `"1"` key types silently drop rows, so normalise on ingest; a single skewed key defeats the in-memory approach and needs a sort-merge or distributed shuffle.

**Interview follow-ups.**

1. *Why does Spark broadcast the small side?* It ships one copy of the small table to every executor, turning a shuffle join into a map-side join with no network shuffle of the fact data.
2. *When is a sort-merge join better than a hash join?* When neither side fits in memory, when both inputs are already sorted, or when the data is skewed and a range partition balances better than a hash.

## Problem 12 - Schema drift and late-arriving dedupe

**Statement.** Two batch-ingestion safety checks. (a) Infer the schema per batch and diff it against a baseline, classifying columns as added, removed, or retyped. (b) Deduplicate replays and out-of-order backfills by keeping the record with the greatest `(updated_at, tiebreak)` per business key.

**Example.**

```text
baseline: {"id":"int", "amount":"float"}
current : {"id":"int", "amount":"str", "coupon":"str"}
drift   : {"added":["coupon"], "removed":[], "retyped":["amount"]}

input : [("u1",t1), ("u2",t1), ("u1",t3), ("u1",t2)]
output: u1 -> the t3 record, u2 -> the t1 record
```

**Constraints.** O(rows x columns) to infer, O(columns) to diff; O(n) time / O(k) memory for dedupe where k is the distinct key count; the tiebreak makes replays deterministic; do not mutate inputs.

### 12a. Infer the observed schema

For every row record the Python type name per column, ignoring `None` (a null carries no type information). The result is `column -> set of observed type names`, so a column that mixes `int` and `float` is visible rather than silently coerced.

This is *observed* schema, not declared schema: it can only find drift in columns that actually appear in the sample. Diffing Parquet/Avro metadata is O(columns) and catches changes in columns with no rows.

In [81]:
from collections.abc import Iterable, Mapping
from typing import Any


def infer_schema(rows: Iterable[Mapping[str, Any]]) -> dict[str, set[str]]:
    """Column -> set of observed Python type names (None contributes nothing)."""
    schema: dict[str, set[str]] = {}
    for row in rows:
        for column, value in row.items():
            if value is not None:
                schema.setdefault(column, set()).add(type(value).__name__)
    return schema

### 12b. Collapse a type set into a label

A column with no non-null value has no type and is labelled `null`; a single type keeps its name; anything else is `mixed(a|b)` in sorted order. Sorting makes the label deterministic, which matters because the drift diff compares labels as strings.

Reporting `mixed(int|float)` is usually benign widening, while a type that becomes `str` is the one that breaks readers.

In [82]:
def describe_type(types: set[str]) -> str:
    """Collapse a set of observed types into one readable label."""
    if not types:
        return "null"
    if len(types) == 1:
        return next(iter(types))
    return "mixed(" + "|".join(sorted(types)) + ")"

### 12c. Diff baseline against current

Set algebra on the column names gives added (`now - base`) and removed (`base - now`); the intersection is checked for a changed type label and reported as retyped. All three lists are sorted so the output is stable and diffable in CI.

The diff is O(columns) once the schemas exist, which is why comparing declared catalog metadata is far cheaper than rescanning every row.

In [83]:
def detect_schema_drift(
    baseline: Mapping[str, set[str]], current: Mapping[str, set[str]]
) -> dict[str, Any]:
    """Classify every column as added, removed, or retyped."""
    base, now = set(baseline), set(current)
    return {
        "added": sorted(now - base),
        "removed": sorted(base - now),
        "retyped": sorted(
            column
            for column in base & now
            if describe_type(baseline[column]) != describe_type(current[column])
        ),
    }

### 12d. Keep the latest record per key

One pass with a dict keyed by the business key. A record replaces the stored winner only when its `(updated_at, tiebreak)` rank is strictly greater, so the first record wins ties -- which is what makes replays deterministic.

The tiebreak (`offset`, version, or ingestion id) is not optional: if two replays carry the same timestamp, a map with no secondary key picks an arbitrary winner and the sink stops converging. Tombstones and deletes must also participate here.

In [84]:
def dedupe_latest(
    records: Iterable[dict[str, Any]],
    key: str = "id",
    ts: str = "updated_at",
    tiebreak: str = "offset",
) -> list[dict[str, Any]]:
    """Keep the record with the greatest (timestamp, tiebreak) per key."""
    winners: dict[Any, dict[str, Any]] = {}
    for record in records:
        current = winners.get(record[key])
        rank = (record[ts], record.get(tiebreak, 0))
        if current is None or rank > (current[ts], current.get(tiebreak, 0)):
            winners[record[key]] = record
    return list(winners.values())

### 12e. Happy path - drift

The documented baseline declares `amount` as a float and `note` as optional; the current batch retypes `amount` to a string and adds `coupon`. The diff must report exactly `{"added": ["coupon"], "removed": [], "retyped": ["amount"]}` -- a string-to-number change is the breaking kind, so it must not be silently widened.

In [85]:
baseline = [{"id": 1, "amount": 10.0, "note": "x"}, {"id": 2, "amount": 3.0, "note": None}]
current = [{"id": 3, "amount": "10", "note": "y", "coupon": "SAVE"}]
drift = detect_schema_drift(infer_schema(baseline), infer_schema(current))
print("drift:", drift)
assert drift == {"added": ["coupon"], "removed": [], "retyped": ["amount"]}
assert detect_schema_drift({}, {}) == {"added": [], "removed": [], "retyped": []}
assert detect_schema_drift({"a": {"int"}}, {"a": {"int"}, "b": {"str"}})["added"] == ["b"]

drift: {'added': ['coupon'], 'removed': [], 'retyped': ['amount']}


### 12f. Happy path - type labels

The label helper is the vocabulary of the drift report, so it gets its own checks: an empty type set is `null`, one type keeps its name, and a mixture is sorted and joined. This is also where an all-null column is distinguishable from a missing column.

In [86]:
assert describe_type(set()) == "null"
assert describe_type({"int"}) == "int"
assert describe_type({"float", "int"}) == "mixed(float|int)"

### 12g. Happy path - dedupe latest wins

Five records for two keys: `u1` has an old row, a stale row with a large offset but an older timestamp, a newest row at `t = 3`, and a `t = 3` tiebreak loser. The winner is the `t = 3` record with offset 9, because the timestamp dominates and the offset only breaks ties.

In [87]:
records = [
    {"id": "u1", "updated_at": 1, "offset": 1, "v": "old"},
    {"id": "u2", "updated_at": 1, "offset": 1, "v": "only"},
    {"id": "u1", "updated_at": 3, "offset": 9, "v": "new"},
    {"id": "u1", "updated_at": 3, "offset": 4, "v": "older-tie"},
    {"id": "u1", "updated_at": 2, "offset": 50, "v": "stale"},
]
result = dedupe_latest(records)
by_id = {row["id"]: row for row in result}
print("winners:", {k: v["v"] for k, v in by_id.items()})
assert by_id["u1"]["v"] == "new"      # greatest timestamp wins
assert by_id["u2"]["v"] == "only"     # single record retained
assert len(result) == 2

winners: {'u1': 'new', 'u2': 'only'}


### 12h. Edge cases - empty input and no mutation

`dedupe_latest([])` returns `[]`, and choosing a winner must not edit the losing or winning records in place -- the function stores references, so the caller's objects are shared but never rewritten. That property is what makes it safe to run over a cached batch twice.

In [88]:
assert dedupe_latest([]) == []
assert records[0]["v"] == "old"       # input not mutated

### 12i. Stated output check

Print the drift report and the winner for each key. The drift dict must match the example verbatim, and the winners map should read `{"u1": "new", "u2": "only"}`.

In [89]:
print("drift   ->", drift)
print("winners ->", {k: v["v"] for k, v in by_id.items()})
print("P12 checks passed")

drift   -> {'added': ['coupon'], 'removed': [], 'retyped': ['amount']}
winners -> {'u1': 'new', 'u2': 'only'}
P12 checks passed


**Complexity & edge cases.** Inference is O(rows x columns); the diff is O(columns), which is why diffing declared Parquet/Avro metadata beats sampling data. Dedupe is O(n) time / O(k) memory; a warehouse `MERGE` or sort-merge spills instead. Edge cases: all-null columns have no type; int-to-float widening is usually benign while string-to-number is breaking; column-order changes are invisible to a set diff but break positional CSV readers; equal timestamps need a version or Kafka-offset tiebreak; naive and aware datetimes cannot be compared, so normalise to UTC; tombstone/delete rows must win and be applied by the writer.

**Interview follow-ups.**

1. *Where should drift detection run - in the producer or the consumer?* Both: producers validate against a contract, consumers compare the landed batch against the catalog because producers lie or get bypassed.
2. *Why is the tiebreak important for idempotency?* Without it, two replays of the same timestamp can pick different winners, so the sink is non-deterministic and reruns do not converge.

## Problem 13 - Sessionize clickstream events

**Statement.** Group ordered user events into sessions, starting a new session when the idle gap exceeds a threshold - the exercise behind every product-analytics model. Handle out-of-order input and per-user interleaving.

**Example.**

```text
events: u1 at [0, 300, 2400, 2500], u2 at [0], gap = 1800 seconds
output: [u1 0-300 (2 events), u1 2400-2500 (2 events), u2 0-0 (1 event)]
```

**Constraints.** Sort by `(user, timestamp)` first; O(n log n) time, O(sessions) output; a gap of exactly the threshold stays in the session; reject a negative gap; a single event forms a one-event session.

### 13a. Sort, then extend the open session

Sorting by `(user, ts, event_id)` makes the problem sequential: each event either extends the last open session for the same user when `when - last_end <= gap`, or opens a new one. The event id is the final tiebreak so equal timestamps produce a deterministic order.

The boundary is inclusive on purpose -- a gap of exactly `gap_seconds` is still the same session, which is how most product-analytics definitions are written. State the choice explicitly, because `<` versus `<=` changes session counts at the edge.

In [90]:
from collections.abc import Iterable
from typing import Any


def sessionize(
    events: Iterable[dict[str, Any]],
    gap_seconds: int,
    user_key: str = "user",
    ts_key: str = "ts",
    id_key: str = "event_id",
) -> list[dict[str, Any]]:
    """Group events into sessions; a new session starts when the idle gap exceeds
    `gap_seconds`. The boundary is inclusive: a gap of exactly `gap_seconds`
    stays in the same session.
    """
    if gap_seconds < 0:
        raise ValueError("gap_seconds must be non-negative")
    ordered = sorted(events, key=lambda event: (event[user_key], event[ts_key], event[id_key]))
    sessions: list[dict[str, Any]] = []
    for event in ordered:
        user, when = event[user_key], event[ts_key]
        if (
            sessions
            and sessions[-1]["user"] == user
            and when - sessions[-1]["end"] <= gap_seconds
        ):
            session = sessions[-1]
            session["end"] = when
            session["count"] += 1
            session["duration"] = session["end"] - session["start"]
            session["event_ids"].append(event[id_key])
        else:
            sessions.append(
                {
                    "user": user,
                    "start": when,
                    "end": when,
                    "count": 1,
                    "duration": 0,
                    "event_ids": [event[id_key]],
                }
            )
    return sessions

### 13b. Happy path

For `gap = 1800`, the documented events produce three sessions: `u1` at 0-300 (two events), `u1` at 2400-2500 (two events, after the 2100-second idle gap), and `u2` at 0-0 (one event). Each session carries its start, end, count, duration, and the ordered event ids.

In [91]:
events = [
    {"user": "u1", "ts": 0, "event_id": "e1"},
    {"user": "u1", "ts": 300, "event_id": "e2"},
    {"user": "u1", "ts": 2400, "event_id": "e3"},  # 2100s gap -> new session
    {"user": "u1", "ts": 2500, "event_id": "e4"},
    {"user": "u2", "ts": 0, "event_id": "e5"},
]
sessions = sessionize(events, gap_seconds=1800)
print("sessions:", [(s["user"], s["start"], s["end"], s["count"]) for s in sessions])
assert [(s["user"], s["start"], s["end"], s["count"]) for s in sessions] == [
    ("u1", 0, 300, 2),
    ("u1", 2400, 2500, 2),
    ("u2", 0, 0, 1),
]
assert sessions[0]["event_ids"] == ["e1", "e2"]
assert sessions[0]["duration"] == 300
assert sessions[2]["duration"] == 0

sessions: [('u1', 0, 300, 2), ('u1', 2400, 2500, 2), ('u2', 0, 0, 1)]


### 13c. Edge cases - inclusive boundary, out-of-order input, empty, negative gap

Exactly the gap keeps the events in one session, and a shuffled input must yield byte-identical output because the function sorts internally before doing anything else. An empty input is a valid empty result, while a negative gap is a caller bug.

Out-of-order input is handled by sorting, but note the limitation: a truly unbounded stream needs a lateness buffer, because events cannot be re-sorted once a session has been emitted.

In [92]:
same = sessionize(
    [{"user": "u", "ts": 0, "event_id": "a"}, {"user": "u", "ts": 1800, "event_id": "b"}], 1800
)
assert len(same) == 1
assert same[0]["count"] == 2

shuffled = [events[3], events[0], events[1], events[2], events[4]]
assert sessionize(shuffled, 1800) == sessions
assert sessionize([], 1800) == []

try:
    sessionize([], -1)
except ValueError:
    pass
else:
    raise AssertionError("negative gap must raise")

### 13d. Stated output check

Print the compact `(user, start, end, count)` view and compare it with the example. The saved output is the fastest way for a reviewer to confirm the session boundaries without reading the assert.

In [93]:
print("sessions ->", [(s["user"], s["start"], s["end"], s["count"]) for s in sessions])
print("ids      ->", [s["event_ids"] for s in sessions])
print("P13 checks passed")

sessions -> [('u1', 0, 300, 2), ('u1', 2400, 2500, 2), ('u2', 0, 0, 1)]
ids      -> [['e1', 'e2'], ['e3', 'e4'], ['e5']]
P13 checks passed


**Complexity & edge cases.** O(n log n) if the input must be sorted, O(n) if the source already delivers ordered partitions per user; memory is O(sessions) for the output (or O(1) state per active user when streaming). Edge cases: out-of-order events need a lateness buffer before emitting; one event forms a one-event session; define `<=` vs `<` at the boundary (here exactly the gap is inactive); users with no events are absent, so downstream joins must be left joins; clock skew can make a user's timestamps non-monotonic.

**Interview follow-ups.**

1. *How would you sessionize an unbounded stream?* Keep per-user last-event state in a keyed store (Redis/Flink state) and emit a session when a timer for `last_event + gap` fires, handling late events with allowed lateness.
2. *How does this map to a Spark window function?* `LAG(ts)` over `PARTITION BY user ORDER BY ts`, then mark a new session when `ts - lag > gap`, and finally `SUM(new_session)` as a session id.

## Problem 14 - Production utilities: LRU cache and retry with backoff

**Statement.** Two small classes you will be asked to write (or reason about) in a data-platform interview: a fixed-capacity **LRU cache** with O(1) `get`/`put`, and a **retry** helper with capped exponential backoff plus optional jitter.

**Example.**

```text
capacity 2: put("a",1) put("b",2) get("a") -> 1; put("c",3) evicts "b"
backoff  : attempts=4, base=0.1, factor=2 -> sleeps [0.1, 0.2, 0.4]
```

**Constraints.** O(1) amortised `get`/`put`; validate capacity; injectable `sleep`/`rand` so the retry logic is testable and deterministic; cap both the delay and the attempt count; re-raise the final error.

### 14a. `LRUCache` on top of `OrderedDict`

`OrderedDict` is a hash map plus a doubly linked list, which is exactly the two data structures an LRU cache needs. A hit calls `move_to_end` to mark the key most-recently-used; a write past capacity evicts with `popitem(last=False)`, the least-recently-used end. Both operations are O(1).

The hit/miss counters make the cache observable in tests. The documented pitfall: `get` returns `None` for a miss, so `None` cannot also be a stored value without a sentinel.

In [94]:
import random
import time
from collections import OrderedDict
from collections.abc import Callable, Hashable
from typing import Generic, TypeVar

K = TypeVar("K", bound=Hashable)
V = TypeVar("V")


class LRUCache(Generic[K, V]):
    """O(1) get/put: an OrderedDict is a hash map plus a doubly linked list."""

    def __init__(self, capacity: int) -> None:
        if capacity <= 0:
            raise ValueError("capacity must be positive")
        self.capacity = capacity
        self._data: OrderedDict[K, V] = OrderedDict()
        self.hits = 0
        self.misses = 0

    def get(self, key: K) -> V | None:
        if key not in self._data:
            self.misses += 1
            return None
        self.hits += 1
        self._data.move_to_end(key)
        return self._data[key]

    def put(self, key: K, value: V) -> None:
        if key in self._data:
            self._data.move_to_end(key)
        self._data[key] = value
        if len(self._data) > self.capacity:
            self._data.popitem(last=False)

    def keys_oldest_first(self) -> list[K]:
        return list(self._data)

### 14b. One delay: `min(cap, base * factor**attempt)`

`compute_delay` is the pure core of the backoff policy: the attempt number goes in, the sleep before the next try comes out, capped by `max_delay` so a long outage does not create a 30-minute sleep. The random source is injected so a test can pin jitter to a known value.

Jitter interpolates between the deterministic delay (`jitter = 0`) and a random one (`jitter = 1`, "full jitter"), which is what desynchronises clients that failed together.

In [95]:
def compute_delay(
    attempt: int,
    base: float,
    factor: float,
    max_delay: float,
    jitter: float,
    rand: Callable[[], float],
) -> float:
    """Delay before the retry after a zero-indexed failed `attempt`."""
    delay = min(max_delay, base * factor**attempt)
    if jitter:
        # Full jitter (jitter=1) spreads retries uniformly; partial jitter dampens.
        delay = delay * (1 - jitter) + delay * jitter * rand()
    return delay

### 14c. The whole schedule

`backoff_delays(attempts)` returns the sleep before each retry for a plan of `attempts` total tries, so there are `attempts - 1` sleeps: a plan of 4 tries sleeps 3 times. Expressing the schedule as a list makes it trivial to assert and to log as a plan before running it.

`attempts = 1` means "try once, never retry" and therefore an empty schedule.

In [96]:
def backoff_delays(
    attempts: int,
    base: float = 0.1,
    factor: float = 2.0,
    max_delay: float = 10.0,
    jitter: float = 0.0,
    rand: Callable[[], float] = random.random,
) -> list[float]:
    """The sleep before each retry for a plan of `attempts` total tries."""
    return [compute_delay(n, base, factor, max_delay, jitter, rand) for n in range(attempts - 1)]

### 14d. The retry loop

Run the callable, catch only the declared exception types, and sleep before the next try; on the last attempt re-raise the original error instead of a wrapper, so the caller sees the real failure. `sleep` and `rand` are injected, which turns a timing-dependent routine into a deterministic one under test.

Retry only transient, idempotent operations: retrying a non-idempotent write double-writes. `attempts < 1` is rejected rather than treated as "no tries".

In [97]:
def retry(
    func: Callable[[], V],
    attempts: int = 4,
    base: float = 0.1,
    factor: float = 2.0,
    max_delay: float = 10.0,
    jitter: float = 0.0,
    retry_on: tuple[type[BaseException], ...] = (Exception,),
    sleep: Callable[[float], None] = time.sleep,
    rand: Callable[[], float] = random.random,
) -> V:
    """Run `func`, retrying transient failures with capped exponential backoff."""
    if attempts < 1:
        raise ValueError("attempts must be >= 1")
    for attempt in range(attempts):
        try:
            return func()
        except retry_on:
            if attempt == attempts - 1:
                raise
            sleep(compute_delay(attempt, base, factor, max_delay, jitter, rand))
    raise RuntimeError("unreachable")

### 14e. Happy path - cache

With capacity 2: put `a`, put `b`, get `a` (which makes `a` the most recent), then put `c` evicts `b`. Updating an existing key keeps it freshest, and the counters report exactly the hits and misses the sequence implies.

In [98]:
cache: LRUCache[str, int] = LRUCache(2)
cache.put("a", 1)
cache.put("b", 2)
assert cache.get("a") == 1          # "a" is now most recently used
cache.put("c", 3)                   # evicts "b"
assert cache.get("b") is None
assert cache.get("c") == 3
assert cache.keys_oldest_first() == ["a", "c"]
cache.put("c", 30)                  # update keeps it freshest
assert cache.get("c") == 30
cache.put("d", 4)                   # evicts "a"
assert cache.get("a") is None
print("cache hits/misses:", cache.hits, cache.misses)
assert cache.hits == 3 and cache.misses == 2

cache hits/misses: 3 2


### 14f. Happy path - backoff schedule and a flaky callable

The documented plan `attempts=4, base=0.1, factor=2` sleeps `[0.1, 0.2, 0.4]`, and a cap of 5.0 flattens a base-3 schedule to `[1.0, 3.0, 5.0, 5.0]`. A function that fails twice then succeeds must be called exactly three times and must sleep exactly `[0.1, 0.2]`.

In [99]:
calls = {"n": 0}
slept: list[float] = []


def flaky() -> str:
    calls["n"] += 1
    if calls["n"] < 3:
        raise TimeoutError("temporary")
    return "ok"


print("delays:", backoff_delays(4, base=0.1, factor=2.0, max_delay=10.0))
assert backoff_delays(4, base=0.1, factor=2.0, max_delay=10.0) == [0.1, 0.2, 0.4]
assert backoff_delays(5, base=1.0, factor=3.0, max_delay=5.0) == [1.0, 3.0, 5.0, 5.0]
assert backoff_delays(1) == []
assert retry(flaky, attempts=5, base=0.1, factor=2.0, sleep=slept.append) == "ok"
assert calls["n"] == 3
print("slept between retries:", slept)
assert slept == [0.1, 0.2]          # two failures -> two sleeps

delays: [0.1, 0.2, 0.4]
slept between retries: [0.1, 0.2]


### 14g. Edge cases - invalid capacity, exhausted retries, pinned jitter

Capacity zero or negative must be rejected, not silently no-op. A callable that always fails must surface the original exception after the final attempt rather than looping forever. Jitter is checked with a pinned random draw: `rand = 0` gives the deterministic lower bound, `rand = 1` the full delay.

In [100]:
try:
    LRUCache(0)
except ValueError:
    pass
else:
    raise AssertionError("capacity 0 must raise")


def always_fails() -> None:
    raise TimeoutError("down")


try:
    retry(always_fails, attempts=2, sleep=slept.append)
except TimeoutError:
    pass
else:
    raise AssertionError("exhausted retries must raise")

assert compute_delay(0, base=1.0, factor=2.0, max_delay=10.0, jitter=0.5, rand=lambda: 0.0) == 0.5
assert compute_delay(0, base=1.0, factor=2.0, max_delay=10.0, jitter=0.5, rand=lambda: 1.0) == 1.0
assert retry(lambda: "ok", attempts=1) == "ok"

### 14h. Stated output check

Print the cache counters, the documented delay schedule, and the observed sleeps. `hits = 3, misses = 2` is exactly the access pattern above: hits on `a`, `c`, and the second `c`; misses on `b` and the evicted `a`.

In [101]:
print("hits/misses ->", cache.hits, cache.misses)
print("schedule    ->", backoff_delays(4, base=0.1, factor=2.0, max_delay=10.0))
print("slept       ->", slept)
print("P14 checks passed")

hits/misses -> 3 2
schedule    -> [0.1, 0.2, 0.4]
slept       -> [0.1, 0.2, 0.1]
P14 checks passed


**Complexity & edge cases.** `OrderedDict` gives O(1) amortised `get`/`put` because it is a hash map plus a doubly linked list; `move_to_end` makes a hit the most recent and `popitem(last=False)` evicts the least recent. Edge cases: capacity zero/negative must be rejected, not silently no-op; `None` as a legal cached value is ambiguous with a miss, so use a sentinel; a hand-rolled version must update both the map and the list on every hit and eviction; retries must target idempotent, transient errors only; cap attempts and `max_delay` so a scheduler slot is not held forever; jitter is mandatory to avoid retry storms.

**Interview follow-ups.**

1. *Why does `functools.lru_cache` not solve every caching need?* It keys on hashable positional/keyword arguments, cannot expire by time, cannot be invalidated selectively, and is per-process rather than shared.
2. *What is full jitter and why does it work?* Sleep `uniform(0, min(cap, base * 2**n))`; randomising waits desynchronises clients that failed together, flattening the thundering herd when the service recovers.

## Interview questions

Twelve questions that come up when interviewers probe Python for data engineering, each with a concise model answer. Where a claim is checkable, a code cell in this section proves it.

### Q1. When do you reach for a generator instead of a list?

**Model answer.** A list materialises every result: O(n) memory and O(n) time up front. A generator produces one item per `next()` and holds O(1) state, so it can represent an unbounded or larger-than-RAM source. Use generators for file/stream ingestion and one-pass transform pipelines; use lists when you need random access, `len()`, reuse, or multiple passes.

### Q2. Why is `dict`/`set` lookup considered O(1), and what breaks it?

**Model answer.** They are hash tables: `hash(key)` picks a bucket, then equality confirms the match. Average-case lookup/insert/delete is O(1); iteration is O(n). Worst case is O(n) per operation under heavy collisions (a bad `__hash__` or adversarial keys). Keys must be hashable with value-based equality; a mutable key would break bucket invariants. Dicts preserve insertion order since Python 3.7, which is what makes an ordered dedupe a one-liner.

### Q3. `sorted()`, `list.sort()`, or `heapq.nlargest()` - which and why?

**Model answer.** `list.sort()` sorts in place, O(n log n) time, O(1) extra space (Timsort may use O(n/2) worst case), returns `None`. `sorted()` copies first, adding O(n) memory. `heapq.nlargest(k, it)` is O(n log k) time and O(k) memory, so it wins when k << n and preserves input order for ties. For top-N per group, keep one bounded heap per group.

### Verification A - generators, heap selection, ordered dicts

Three checkable claims from Q1-Q3. The heap comparison proves `nlargest` agrees with a full sort while bounding memory; the generator size comparison proves a generator's state does not grow with its potential output; and the dict checks show first-seen key order with last-write-wins values.

In [102]:
import heapq
import sys

# nlargest equals a full-sort slice, but keeps memory bounded by k.
data = [5, 1, 9, 3, 7, 2]
assert heapq.nlargest(3, data) == sorted(data, reverse=True)[:3]
assert heapq.nsmallest(2, data) == sorted(data)[:2]


# A generator holds constant state regardless of how many items it could produce.
def count_up(limit):
    value = 0
    while value < limit:
        yield value
        value += 1


large_generator = count_up(10**9)
assert sys.getsizeof(large_generator) < sys.getsizeof([0] * 10_000)
assert [next(large_generator) for _ in range(3)] == [0, 1, 2]
large_generator.close()


# dict preserves insertion order, which is why ordered dedupe is easy.
pairs = [("a", 1), ("b", 2), ("a", 3)]
assert list(dict(pairs)) == ["a", "b"]     # first-seen order
assert dict(pairs)["a"] == 3               # but last write wins the value
assert list(dict.fromkeys(["x", "y", "x"])) == ["x", "y"]
print("Interview verification A passed")

Interview verification A passed


### Q4. How do you deduplicate records while preserving order, and how does it scale?

**Model answer.** Stream the data and keep a `set` of seen keys, appending the first occurrence of each: O(n) time, O(k) memory for k distinct keys. At scale, partition by key so each partition dedupes independently; or sort by key and drop adjacent duplicates (spills to disk, O(n log n)); or use a Bloom filter as a cheap pre-filter, accepting false positives. In a warehouse: `ROW_NUMBER() OVER (PARTITION BY key ORDER BY ts DESC) = 1`.

### Q5. How do you process a file larger than RAM with only the standard library?

**Model answer.** Never call `read()`/`readlines()`. Iterate the file object line by line (bounded buffer), wrap it in a generator, process in fixed batches (`itertools.batched`), and keep only aggregates or the required state. Use `gzip.open(..., "rt")` for compressed files. Parallelise by byte ranges only when the format allows seeking to a record boundary; otherwise split by file. Write aggregates out incrementally.

### Q6. `json.load` vs `json.loads` vs streaming with `raw_decode`.

**Model answer.** `json.loads(str)` parses a string; `json.load(file)` reads a file object and parses it, but both materialise the whole document. For a JSON array too large for memory, drive `json.JSONDecoder().raw_decode(buffer, index)` to pull one value at a time, or use NDJSON where every line is an independent document. Third-party options are `ijson` and `jsonlines`.

### Q7. How does an in-memory hash join work, and when does it fail?

**Model answer.** Build a hash map on the smaller (dimension) side, then stream the larger (fact) side through it: O(n + m) time, O(m) memory for the build side. It fails when the build side does not fit in memory, when one join key is badly skewed, or when null/type semantics differ (`1` vs `"1"`, NULL never equals NULL). Remedies: broadcast the small side, sort-merge join, or hash-partition by key and join partition-wise.

### Q8. How do you handle late-arriving records and duplicates?

**Model answer.** Make the sink idempotent. Keep a map keyed by the business key and retain the record with the greatest `(event_timestamp, tiebreak)`, where the tiebreak is a Kafka offset or version so replays are deterministic. If producer clocks are unreliable, prefer the ingestion timestamp for lateness. Tombstones/deletes must also win, and the downstream write must merge/upsert rather than blindly append.

### Q9. What is schema drift and how do you detect it?

**Model answer.** Schema drift is any change in columns or types between batches. Infer the observed schema (column -> set of types) per batch, compare against the catalog or the previous batch, and classify added, removed, and retyped columns. Silent type changes such as int -> string are the dangerous ones. Diff declared Parquet/Avro metadata in O(columns); added required or removed fields break readers, while int -> float is usually benign widening.

### Verification B - recency order, hash-join fan-out, sorted invariants

Three checkable claims from Q7-Q9. `OrderedDict.move_to_end` plus `popitem(last=False)` is the LRU primitive; a hash join with duplicate build keys multiplies rows exactly like SQL; and unsorted input breaks the binary-search partition invariant, which is why the check is explicit in Problem 7.

In [103]:
from collections import OrderedDict, defaultdict

# OrderedDict.move_to_end is the primitive behind LRU recency ordering.
od = OrderedDict()
for key in "abc":
    od[key] = key.upper()
od.move_to_end("a")
assert list(od) == ["b", "c", "a"]         # "a" touched most recently
od.popitem(last=False)
assert list(od) == ["c", "a"]              # oldest evicted

# A hash join with duplicate build keys multiplies rows, exactly like SQL.
build = defaultdict(list)
for key, value in [("k", 1), ("k", 2), ("x", 9)]:
    build[key].append(value)
probe = [("k", "A"), ("x", "B"), ("missing", "C")]
joined = [(key, payload, dim) for key, payload in probe for dim in build.get(key, [])]
assert joined == [("k", "A", 1), ("k", "A", 2), ("x", "B", 9)]


# Rejecting unsorted input is what keeps a binary-search partition valid.
def kth_partition_is_valid(sorted_values):
    return all(sorted_values[i] <= sorted_values[i + 1] for i in range(len(sorted_values) - 1))


assert kth_partition_is_valid([1, 2, 2, 5])
assert not kth_partition_is_valid([2, 1])
print("Interview verification B passed")

Interview verification B passed


### Q10. How do you compute percentiles and rolling windows efficiently?

**Model answer.** An exact percentile needs the sorted sample: sort once (O(n log n)) and each percentile is O(1). For rolling windows, means/sums are O(1) per element with a deque, but exact quantiles are not. Use an order-statistic tree (O(log n) insert) or an approximate sketch (t-digest, HDR histogram, Greenwald-Khanna) for large windows. Percentiles are not additive across buckets, and on a small sample p99 is just the maximum.

### Q11. How do you implement an LRU cache, and what does `functools.lru_cache` give you?

**Model answer.** Combine a hash map (O(1) lookup) with a doubly linked list or `OrderedDict` (recency order): `get` moves the key to the most-recent end, `put` inserts and evicts the least-recent end when over capacity. Both are O(1). `functools.lru_cache` is the C implementation for pure functions keyed on hashable arguments; it cannot key unhashable objects, has no time-based expiry, and caches per process. A sentinel is needed when `None` is a legal value.

### Q12. Why retry with exponential backoff and jitter?

**Model answer.** Delay before retry n is `min(cap, base * factor**n)`, plus jitter. Backoff gives a failing dependency time to recover; jitter (`uniform(0, delay)` for full jitter) desynchronises clients so they do not stampede the service the moment it returns. Retry only transient, idempotent operations (timeouts, 429, 503), cap attempts and max delay so no scheduler slot is held forever, and re-raise the final error after recording it.

### Verification C - memoization, streaming JSON, backoff arithmetic

Three checkable claims from Q10-Q12. `functools.lru_cache` exposes its hit/miss counters and evicts by capacity; `raw_decode` returns the index where one document stopped, which is the primitive behind streaming JSON; and the backoff delay is a pure function of the attempt number, so it can be asserted exactly.

In [104]:
import functools
import json

# functools.lru_cache memoizes by hashable args; cache_info exposes the counters.
calls = {"n": 0}


@functools.lru_cache(maxsize=2)
def slow_square(value):
    calls["n"] += 1
    return value * value


assert [slow_square(v) for v in (1, 2, 1, 3, 1)] == [1, 4, 1, 9, 1]
assert calls["n"] == 3                    # only 1, 2, 3 were computed
info = slow_square.cache_info()
assert (info.hits, info.misses) == (2, 3)

# raw_decode parses one value at a time out of a buffer: the streaming JSON core.
decoder = json.JSONDecoder()
text = '{"a": 1} {"b": 2}'
first, index = decoder.raw_decode(text)
second, _ = decoder.raw_decode(text, index + 1)
assert (first, second) == ({"a": 1}, {"b": 2})

# Exponential backoff is a pure function of the attempt number.
delays = [min(0.5 * 2**n, 4.0) for n in range(5)]
assert delays == [0.5, 1.0, 2.0, 4.0, 4.0]
print("Interview verification C passed")

Interview verification C passed


### Closing checklist

Before you call a coding answer finished in an interview, walk this list out loud:

- **State the preconditions.** Sorted input, non-null keys, hashable types, ascending partitions.
- **State the complexity** for time and memory, including the build side of a join or the retained state of a stream.
- **Name the edge cases** you handle and the ones you deliberately reject with an error.
- **Say what changes at scale**, and which single-machine assumption breaks first.